In [ ]:
!rm -rf /content/faiss_index /content/faiss_index_cbr

In [ ]:
!rm -f /content/checkpoint_llama31-8b-q8__grounded-guard.jsonl
!rm -f "/content/drive/MyDrive/SCJN_RAG/results/RL_LLM__llama31-8b-q8/checkpoint_llama31-8b-q8__grounded-guard.jsonl"

In [ ]:
# -*- coding: utf-8 -*-
"""
================================================================================
  RL + LLM Legal Pipeline  —  v1.0  TEST-TIME RL MULTI-MODEL BENCHMARK
================================================================================
  Doctoral Research Program · Predictive Model · Mexican Legal Sector (SCJN)
  Corpus       : SCJN Resolutions (PDF) + 9 Mexican Labor Law PDFs
  Case bank    : gold_prediction_90_compatible.json  (90 reference cases, used
                 here as a Case-Based-Reasoning [CBR] exemplar bank, NOT as a
                 held-out test set — see §PRE and §6b)
  Rubric       : rubrica_validacion_contratos.json   (20 validation questions,
                 doubles as the reward-model prompt bank — see §7)
  Environment  : Google Colab T4 GPU · Python 3.12

  OBJECTIVE
  ─────────
  Produce and evaluate a Spanish legal-recommendation assistant that generates
  a litigation STRATEGY for labor-contract cases under Mexican law, and
  identify — via a controlled benchmark — which base model best serves that
  objective inside an RL+LLM pipeline, where RL is implemented as TEST-TIME
  reward-guided rollout selection and self-refinement (Modo A — no gradient
  updates to the candidate models; see design rationale below).

  RL DESIGN — MODO A (validated with the user, doctoral thesis v1)
  ──────────────────────────────────────────────────────────────
  Candidate policies are frozen GGUF checkpoints (llama.cpp) with no autograd
  support, so classical policy-gradient RL (PPO/GRPO weight updates) is not
  applied to them in this script. Instead, RL is implemented as:
    1. Best-of-N sampling  : N stochastic rollouts per case from the candidate
    2. Reward scoring       : composite reward (rubric LLM-as-judge +
                              article-grounding F1 + CBR outcome alignment)
    3. Self-refinement      : 1 revision round on the best rollout, guided by
                              the reward model's structured feedback
  This is consistent with the test-time / inference-time RL literature
  (reward-guided decoding, RLAIF-style AI feedback as the reward signal —
  Lee et al., 2023, arXiv:2309.00267) without requiring GRPO/PPO fine-tuning
  infrastructure (Shao et al., 2024, arXiv:2402.03300), which is reserved for
  a Phase-2 fine-tune of the single winning model outside this benchmark.

  JUDGE DESIGN — fixed, independent, out-of-family
  ─────────────────────────────────────────────────
  The reward model's rubric component uses a FIXED judge model
  (Phi-3.5-mini-instruct, Q4_K_M) that is architecturally unrelated to every
  candidate family (Llama-3.1, Qwen2.5, Mistral-v0.3, Gemma-2), and is NEVER
  assigned as MODEL_KEY in the benchmark loop. This mitigates self-enhancement
  bias, documented as a systematic failure mode of the LLM-as-judge paradigm
  (Zheng et al., 2023, arXiv:2306.05685).

  EXPERIMENTAL DESIGN (controlled comparison, single factor = MODEL_KEY)
  ────────────────────────────────────────────────────────────────────
  Five sequential candidate runs (JUDGE_MODEL is never one of these):
      llama31-8b-q4 · qwen25-7b-q4 · mistral-7b-v03-q4 · gemma2-9b-q4 ·
      llama31-8b-q8
  ARM = f"RL_LLM__{MODEL_KEY}" → each run writes its own artifact namespace.

  RUN PLAN — 6 runs total, NOT 10
  ───────────────────────────────
  (A) PRIMARY BENCHMARK — 5 runs, one per MODEL_KEY.
      MODEL_KEY is the ONLY thing that changes. These must stay FROZEN
      across all five arms, or the model comparison is confounded:
          REFINEMENT_FEEDBACK_MODE = "grounded"
          REFINEMENT_ACCEPT_GUARD  = True
          PILOT_MODE               = False   (full 60-case bank)
          N_ROLLOUTS, REWARD_WEIGHTS, TOP_K_*, JUDGE_KEY, prompts
      Answers: which base model best serves the objective?

  (B) REFINEMENT ABLATION — 1 additional run, on ONE model only.
      Re-run a single MODEL_KEY (suggested: the winner of A) with
          REFINEMENT_FEEDBACK_MODE = "intrinsic"
      This is a question about the METHOD, not about the models, so it does
      not need to be repeated across all five arms. RUN_SUFFIX keeps its
      checkpoint and artifacts separate from the (A) run of the same model.
      Answers: does externally-grounded feedback outperform intrinsic
      self-correction (Huang et al., 2024)?

  STATISTICS — two distinct questions, two distinct tools
  ───────────────────────────────────────────────────────
  · ACROSS the 5 models (which model is best): compare Tier-1 metrics via
    their 95% bootstrap CIs (§10). A paired significance test across only
    5 arms would not be appropriate.
  · WITHIN one model (does refinement help): paired Wilcoxon signed-rank on
    reward_refined_raw vs reward_best_of_n over the 60 CASES — n=60 pairs,
    not n=5 models. Prints "skipped" on 5-case pilots by design.

  EVALUATION (extends the 3-tier hierarchy with RL-specific metrics)
  ────────────────────────────────────────────────────────────────
  Tier 1 (headline)  : 3-class decision accuracy, macro-F1, MCC + 95% CIs.
  Tier 2 (grounding) : article precision/recall/F1 vs. gold citations.
  Tier 3 (secondary) : ROUGE-1/2/L, BLEU, BERTScore-F1.
  RL-specific        : reward_mean/std per arm; win-rate of "refined" vs.
                        "best-of-1" baseline, judged with randomized A/B order
                        to control position bias (Zheng et al., 2023).
  Strategy quality   : 20-question rubric, scored ONLY by JUDGE_MODEL.
  Qualitative        : 5 representative cases for thesis narrative (§12b).

  PIPELINE (run once per MODEL_KEY, top to bottom):
    §0   ABI reset & kernel restart                        (auto)
    §1   Imports
    §PRE Pre-flight checks
    §2   Global constants + MODEL_REGISTRY (5 candidates + 1 fixed judge)
    §3   Embeddings + judge manager init (candidate load DEFERRED to §6c —
         keeps peak RAM lower during the PDF ingestion/embedding phase)
    §4   Helper functions
    §5-6 PDF ingestion + RAG vector index, STREAMED one file at a time
         (FAISS, document corpus — shared across arms)
    §6b  CBR vector index (FAISS, 60-case exemplar bank — shared across arms)
    §6c  Candidate GGUF load (deferred from §3; not needed until §7/§9)
    §7   Reward model: rubric judge + article grounding + CBR alignment
    §8   Rollout generator (Best-of-N stochastic sampling)
    §9   RL-lite loop: generate → score → select → self-refine → rescore
    §10  Evaluation: Tier-1/2/3 metrics + bootstrap CIs + RL metrics
    §11  Litigation Strategy (final structured RAG+CBR+RL output)
    §11b Rubric Evaluation (20-question, JUDGE_MODEL only)
    §12  Visualisation + §12b qualitative 5-case sample
    §13  Save all artifacts to Drive (ARM-namespaced)
    §14  Case Analysis Module (reusable PhD experiment API)

  REFERENCES (APA 7)
  ───────────────────
  Lee, H., Phatale, S., Mansoor, H., Lu, K., Mesnard, T., Bishop, C.,
    Carbune, V., & Rastogi, A. (2023). RLAIF: Scaling reinforcement learning
    from human feedback with AI feedback (arXiv:2309.00267). arXiv.
  Shao, Z., Wang, P., Zhu, Q., Xu, R., Song, J., Zhang, M., Li, Y. K., Wu, Y.,
    & Guo, D. (2024). DeepSeekMath: Pushing the limits of mathematical
    reasoning in open language models (arXiv:2402.03300). arXiv.
  Zheng, L., Chiang, W.-L., Sheng, Y., Zhuang, S., Wu, Z., Zhuang, Y., Lin, Z.,
    Li, Z., Li, D., Xing, E., Zhang, H., Gonzalez, J. E., & Stoica, I. (2023).
    Judging LLM-as-a-judge with MT-Bench and Chatbot Arena. Advances in
    Neural Information Processing Systems, 36, 46595–46623.

  HOW TO RUN
  ──────────
  1. Place all 5 candidate GGUFs + Phi-3.5-mini-instruct-Q4_K_M.gguf in
     Drive/MyDrive/SCJN_RAG/models/
  2. Set MODEL_KEY in §2 (§PRE will refuse to proceed if MODEL_KEY ==
     JUDGE_KEY, or if the judge GGUF is missing)
  3. Run top to bottom. Repeat for each of the 5 MODEL_KEY values.
================================================================================
"""


In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# §0 · ENVIRONMENT SETUP  (v13 parity — replaces the legacy ABI-reset block)
#
# WHY THIS REPLACED THE LEGACY §0 — and why this arm was the last to get it:
#   The legacy §0 performed an "ABI reset": it pinned numpy/scipy, force-
#   reinstalled llama-cpp-python, wrote a sentinel and then KILLED THE KERNEL so
#   the pinned stack would reload. That was correct for the ~2024 Colab image.
#   It is now actively harmful: current Colab ships torch 2.11.0+cu128 with
#   torchvision/torchaudio matched and CUDA available, and every scientific
#   dependency imports cleanly. Downgrading produces a stack that no longer
#   matches the host driver, causing kernel terminations with no Python
#   traceback (ABI-mismatched C extensions segfault — uncatchable).
#
#   The bare-LLM, RAG and GraphRAG arms were migrated to this additive setup and
#   run without any restart. This arm retained the legacy block, which is the
#   sole reason it forces a kernel restart that the other three do not. Behaviour
#   is now identical across all four arms.
#
# NO KERNEL RESTART. Nothing is uninstalled or downgraded, so there is no stale
# ABI to reload. If Colab offers a restart after the installs, decline it.
#
# llama-cpp-python wheel: upstream publishes prebuilt CUDA wheels for
#   cu121-cu125/cu13x but NOT cu128, so this VM cannot be index-matched exactly.
#   cu121 works because CUDA minor-version compatibility lets a 12.1 binary run
#   on a 12.8 driver, and llama-cpp-python links the SYSTEM CUDA runtime (it need
#   not match torch's cu128). --only-binary forces a LOUD failure instead of a
#   silent CPU-only source build, which would make the 90-case rollout loop —
#   the most compute-intensive arm — untenably slow.
# ──────────────────────────────────────────────────────────────────────────────
import subprocess, sys, importlib, os

print("§0  Environment setup (no ABI reset, no kernel restart)...")

_GPU = subprocess.run(["nvidia-smi"], capture_output=True).returncode == 0
_HAS_GPU = _GPU          # exported name that §PRE/§3/§9 depend on
_GPU_NAME = ""
if _HAS_GPU:
    try:
        _GPU_NAME = subprocess.run(
            ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
            capture_output=True, text=True).stdout.strip().split("\n")[0]
    except Exception:
        _GPU_NAME = "GPU"
print(f"§0  GPU detected: {_GPU_NAME if _HAS_GPU else 'no — CPU runtime'}")

# Package set matches the legacy list for this arm (adds networkx/spacy for
# parity with the shared helper layer).
_PKGS = ["langchain", "langchain-community", "langchain-text-splitters",
         "sentence-transformers", "faiss-cpu", "pypdf", "PyPDF2",
         "evaluate", "rouge_score", "bert-score", "sacrebleu",
         "scikit-learn", "pandas", "matplotlib", "seaborn", "networkx", "spacy"]
print(f"§0  Installing {len(_PKGS)} pipeline packages (~2-3 min)...")
_r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_PKGS],
                    capture_output=True, text=True)
print("  ✅ pipeline packages installed" if _r.returncode == 0
      else f"  ⚠️  pip returned {_r.returncode}:\n{_r.stdout[-1200:]}\n{_r.stderr[-1200:]}")

try:
    import llama_cpp
    print(f"  ✅ llama_cpp already present ({llama_cpp.__version__})")
except ModuleNotFoundError:
    print("§0  Installing llama-cpp-python (prebuilt CUDA wheel)...")
    if _HAS_GPU:
        _w = subprocess.run(
            [sys.executable, "-m", "pip", "install", "llama-cpp-python",
             "--prefer-binary", "--only-binary=llama-cpp-python",
             "--extra-index-url",
             "https://abetlen.github.io/llama-cpp-python/whl/cu121"],
            capture_output=True, text=True)
        if _w.returncode != 0:
            print("  ⚠️  cu121 wheel unavailable → retrying cu125...")
            _w = subprocess.run(
                [sys.executable, "-m", "pip", "install", "llama-cpp-python",
                 "--prefer-binary", "--only-binary=llama-cpp-python",
                 "--extra-index-url",
                 "https://abetlen.github.io/llama-cpp-python/whl/cu125"],
                capture_output=True, text=True)
        print("  ✅ llama-cpp-python installed" if _w.returncode == 0
              else f"  ❌ FAILED — do NOT proceed:\n{_w.stdout[-1200:]}\n{_w.stderr[-1200:]}")
    else:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        "llama-cpp-python"], capture_output=True)

_sp = subprocess.run([sys.executable, "-c",
                      "import spacy; spacy.load('es_core_news_lg'); print('ok')"],
                     capture_output=True, text=True)
if "ok" not in _sp.stdout:
    print("§0  Downloading spaCy es_core_news_lg (~600 MB)...")
    subprocess.run([sys.executable, "-m", "spacy", "download",
                    "es_core_news_lg", "--quiet"], capture_output=True)

print("\n§0  Verifying imports required by §1:")
_CHECK = ["torch", "torchvision", "torchaudio", "numpy", "pandas", "matplotlib",
          "seaborn", "networkx", "spacy", "llama_cpp", "langchain_community",
          "langchain_text_splitters", "evaluate", "bert_score", "sklearn"]
_missing = []
for _m in _CHECK:
    try:
        importlib.import_module(_m)
    except Exception as _e:
        _missing.append((_m, type(_e).__name__))
if _missing:
    print(f"  ❌ still missing/broken: {_missing}")
    print("  Do not run §1 yet — report this list.")
else:
    print(f"  ✅ all {len(_CHECK)} modules import cleanly — §1 is safe to run.")


In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# §1  IMPORTS
# ──────────────────────────────────────────────────────────────────────────────
print("§1  Importing packages...")
import json, re, io, warnings, time as _time, fnmatch as _fnmatch, random, traceback
import shutil as _shutil_model
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from llama_cpp import Llama

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

import evaluate as hf_evaluate
from bert_score import score as bert_score_compute
from sklearn.metrics import (classification_report, confusion_matrix,
                             accuracy_score, f1_score, matthews_corrcoef)

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    _IN_COLAB = True
    print("  Google Drive mounted.")
except ImportError:
    _IN_COLAB = False
print("✅  Imports successful.\n")


In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# §2  GLOBAL CONSTANTS + MODEL REGISTRY (5 candidates + 1 fixed judge)
# ──────────────────────────────────────────────────────────────────────────────
DRIVE_ROOT       = "/content/drive/MyDrive/SCJN_RAG"
MODELS_DIR_DRIVE = os.path.join(DRIVE_ROOT, "models")
MODELS_DIR_LOCAL = "/content/models"
RAW_PDFS_PATH    = os.path.join(DRIVE_ROOT, "data/scjn_resoluciones")
LAW_PDFS_PATH    = os.path.join(DRIVE_ROOT, "data/leyes_mexicanas")
EVAL_JSON_PATH   = os.path.join(DRIVE_ROOT, "data/eval_json")
FAISS_INDEX_PATH = os.path.join(DRIVE_ROOT, "faiss_index")          # doc RAG
CBR_INDEX_PATH   = os.path.join(DRIVE_ROOT, "faiss_index_cbr")      # 90-case bank
RUBRIC_JSON_PATH = os.path.join(EVAL_JSON_PATH, "rubrica_validacion_contratos.json")
GOLD_JSON_PATH   = os.path.join(EVAL_JSON_PATH, "gold_prediction_90_compatible.json")
LOCAL_RAW_PDFS   = "/content/raw_pdfs"
LOCAL_LAW_PDFS   = "/content/law_pdfs"

# HARMONIZED with the RAG and GraphRAG arms (identical values in both). The
# document FAISS index is a SHARED, FROZEN artifact: chunking is applied only
# at build time, so these constants are inert while the canonical index is
# loaded read-only. They are aligned anyway so that an accidental rebuild
# cannot silently produce a different corpus than the other arms retrieve from.
CHUNK_SIZE_PREC, CHUNK_OVERLAP_PREC = 2000, 200
CHUNK_SIZE_LAW,  CHUNK_OVERLAP_LAW  = 2400, 300
N_CTX = 8192

# ── RL / reward hyper-parameters ──────────────────────────────────────────────
N_ROLLOUTS       = 5          # Best-of-N per case
ROLLOUT_TEMP     = 0.85
ROLLOUT_TOP_P    = 0.92
REFINE_TEMP      = 0.4        # lower temperature for the self-refinement pass
REWARD_WEIGHTS   = {"rubric": 0.55, "article_grounding": 0.25, "cbr_alignment": 0.20}
TOP_K_RAG        = 4     # HARMONIZED (was 6). Retrieval depth is part of the
                         # frozen pipeline; RAG and GraphRAG both use k=4, so
                         # RL retrieving 6 chunks was an uncontrolled advantage
                         # not attributable to the RL mechanism.
TOP_K_CBR        = 3          # analogous cases retrieved from the 60-case bank

# ── Self-refinement design (see §8) ──────────────────────────────────────────
# Empirical trigger: in the pilot, refined outputs scored BELOW best-of-N
# (e.g. 0.788 -> 0.678). This reproduces the finding of Huang et al. (2024,
# ICLR), who show that INTRINSIC self-correction — revising using only the
# model's own judgement, with no external signal — fails to improve and often
# degrades performance. Tyen et al. (2024) localise the bottleneck precisely:
# models can repair an error once its LOCATION is supplied, but cannot
# reliably find that location themselves.
#
# The v1 prompt asked the candidate to "identificar UNA debilidad concreta"
# — i.e. to locate its own error — while passing only aggregate reward
# scalars. The pipeline meanwhile ALREADY computes exact error locations
# (which rubric criteria failed, which gold articles are missing, whether the
# decision contradicts analogous cases) inside composite_reward and threw
# them away. Two corrections follow:
#
# REFINEMENT_FEEDBACK_MODE
#   "grounded" (default) — inject the located failures into the critique
#       prompt, converting intrinsic self-correction into externally-grounded
#       repair, the regime where the literature reports gains.
#   "intrinsic"          — v1 behaviour (no located feedback). Keep for the
#       ablation arm; expected to underperform.
#
# REFINEMENT_ACCEPT_GUARD
#   True (default) — accept the revision only if its reward improves,
#       otherwise keep best-of-N. Makes the loop monotonically
#       non-decreasing by construction.
#   NOTE ON RIGOUR: this selects by the REWARD MODEL, never by the gold
#       label. It is therefore NOT the "oracle-guided filtering" that Huang
#       et al. identify as the methodological flaw in earlier
#       self-correction papers — the reward model is available at deployment
#       time, gold labels are not, so this remains a legitimate deployable
#       policy. §10 nonetheless reports the UNGUARDED win-rate separately,
#       because that, not the guarded reward, is the honest measurement of
#       whether refinement helps.
REFINEMENT_FEEDBACK_MODE = "grounded"   # "grounded" | "intrinsic"
REFINEMENT_ACCEPT_GUARD  = True
MAX_FEEDBACK_CRITERIA    = 6            # worst-scoring criteria fed back

# ── JUDGE — fixed, independent, never a benchmark candidate ──────────────────
JUDGE_KEY  = "phi35-mini-q4"
JUDGE_SPEC = {"pattern": "*Phi-3.5-mini-instruct*Q4_K_M*.gguf",
              "arch": "Phi-3.5", "params_b": 3.8, "quant": "Q4_K_M",
              "chat_format": None, "n_ctx": 6144, "supports_system_role": True,
              # CPU JUDGE (n_gpu_layers=0). The judge must not compete for VRAM
              # with a resident candidate — with the Q8_0 backbone (~8.5 GB) on
              # a 16 GB accelerator the allocation fails and llama.cpp returns
              # NULL, reported as the misleading "Failed to load model from
              # file". Pinning to CPU also holds the measuring instrument
              # constant across all 20 conditions.
              "n_gpu_layers": -1}
# ── CORRECTIVE FIX · judge device, ARM-SPECIFIC and deliberate ───────────────
# The judge runs on GPU in THIS arm and on CPU in the single-pass arms. That is
# not an oversight; it is forced by a 2-order-of-magnitude difference in call
# volume:
#   RAG / GraphRAG / bare-LLM : judge invoked ONCE per run (§11b demo case).
#                               CPU cost ~seconds. Irrelevant.
#   RL                        : judge invoked per rollout per case as a reward
#                               component — ~30 calls/case x 90 cases ~= 2,700
#                               calls per model.
# Measured: with the judge on CPU this arm ran at ~1,900 s/case => ~47 h for one
# backbone and ~10 days for the matrix. With the judge on GPU the same arm
# previously ran at ~400-450 s/case. The CPU configuration is not merely slower,
# it is infeasible.
#
# Effect on the instrument: judging is greedy (temperature=0.0), so device
# affects only floating-point reduction order, not sampling. Outputs are
# expected to be identical or near-identical. This is a disclosed deviation
# (report it in Methods alongside the per-arm generation budget), and it is a
# far weaker threat to validity than not running the arm at all.
              # n_ctx sized for GROUPED rubric calls (see rubric_batched_judge:
              # RUBRIC_GROUP_SIZE questions per call), NOT the whole rubric at
              # once. Budget per call: rubric group with guidelines ~1.4K tok +
              # facts + strategy (both truncated, see _JUDGE_FACTS_MAX_CHARS /
              # _JUDGE_STRATEGY_MAX_CHARS) + 900 output tokens.
              #
              # CRITICAL — do not raise this casually. Phi-3.5-mini has NO GQA
              # (32 KV heads, head_dim 96), so its KV cache costs ~393 KB/token
              # vs ~131 KB/token for Llama-3.1-8B. On a 15 GB T4 with the
              # candidate resident (~6.0 GB at n_ctx=8192):
              #   judge n_ctx=16384 -> 8.84 GB -> 14.82 GB total -> FAILS with
              #                        "ValueError: Failed to create llama_context"
              #   judge n_ctx= 8192 -> 5.62 GB -> 11.59 GB total -> fits
              #   judge n_ctx= 6144 -> 4.82 GB -> 10.79 GB total -> fits (chosen)
              #   judge n_ctx= 4096 -> 4.01 GB ->  9.98 GB total -> fits

# ── MODEL_KEY — the ONLY variable across the 5 controlled runs ───────────────
MODEL_KEY = "llama31-8b-q4"   # ← change per run; must never equal JUDGE_KEY

MODEL_REGISTRY = {
    "llama31-8b-q4":     {"pattern": "*Llama-3.1-8B-Instruct-Q4_K_M*.gguf",
                          "arch": "Llama-3.1", "params_b": 8, "quant": "Q4_K_M",
                          "chat_format": None, "n_ctx": None, "supports_system_role": True},
    "qwen25-7b-q4":      {"pattern": "*Qwen2.5-7B-Instruct*Q4_K_M*.gguf",
                          "arch": "Qwen2.5", "params_b": 7, "quant": "Q4_K_M",
                          "chat_format": None, "n_ctx": None, "supports_system_role": True},
    "mistral-7b-v03-q4": {"pattern": "*Mistral-7B-Instruct-v0.3*Q4_K_M*.gguf",
                          "arch": "Mistral-v0.3", "params_b": 7, "quant": "Q4_K_M",
                          "chat_format": None, "n_ctx": None, "supports_system_role": True},
    "gemma2-9b-q4":      {"pattern": "*gemma-2-9b-it*Q4_K_M*.gguf",
                          "arch": "Gemma-2", "params_b": 9, "quant": "Q4_K_M",
                          "chat_format": None, "n_ctx": None, "supports_system_role": False},
    "llama31-8b-q8":     {"pattern": "*Llama-3.1-8B-Instruct-Q8_0*.gguf",
                          "arch": "Llama-3.1", "params_b": 8, "quant": "Q8_0",
                          "chat_format": None,
                          # CORRECTIVE FIX: Q8_0 weights (~2x the VRAM of the
                          # Q4_K_M candidates that share this global N_CTX=8192)
                          # + the resident fixed judge caused a hard CUDA OOM
                          # abort on a T4 (ggml_cuda_pool_vmm alloc failure,
                          # kernel restart) during §9's full 90-case run. This
                          # per-model override trims the KV-cache budget only
                          # for this one candidate — it is an inference-engine
                          # resource setting, not part of the experimental
                          # treatment, and is reported alongside quant/arch in
                          # every output JSON for reproducibility.
                          "n_ctx": 6144, "supports_system_role": True},
}

assert MODEL_KEY != JUDGE_KEY, (
    "MODEL_KEY cannot equal JUDGE_KEY — the judge must stay out of the "
    "benchmark loop to avoid self-enhancement bias (Zheng et al., 2023).")
assert MODEL_KEY in MODEL_REGISTRY, f"Unknown MODEL_KEY '{MODEL_KEY}'."

MODEL_SPEC    = MODEL_REGISTRY[MODEL_KEY]
# CORRECTIVE FIX: §2's identity printout was displaying the hardcoded global
# N_CTX (8192) regardless of any per-model MODEL_REGISTRY["n_ctx"] override —
# purely a stale-display bug (the actual load call in §6c already resolved
# the override correctly via _load_gguf's own priority chain: n_ctx_override
# or spec.get("n_ctx") or global_n_ctx). This computes the SAME resolution
# up front so what gets printed matches what will actually be loaded.
N_CTX_EFFECTIVE = MODEL_SPEC.get("n_ctx") or N_CTX
MODEL_ARCH    = MODEL_SPEC["arch"]
MODEL_PARAMS_B= MODEL_SPEC["params_b"]
MODEL_QUANT   = MODEL_SPEC["quant"]
ARM           = f"RL_LLM__{MODEL_KEY}"
ARM_SUFFIX    = MODEL_KEY

# ── §9 execution mode: pilot timing estimate vs. full 90-case run ────────────
# Rationale: N_ROLLOUTS=5 stochastic generations + 1 self-refinement pass per
# case is materially more expensive than v9a's single-shot inference, so we
# time a small pilot BEFORE committing to the full case bank on a T4.
PILOT_MODE       = False     # ← set to False only after reviewing the §9 pilot
                            #   projection printed below; then re-run §9.
PILOT_N_CASES    = 5        # 3-5 recommended; 5 gives a steadier per-case mean
CHECKPOINT_EVERY = 1        # persist after every N cases (1 = safest for Colab
                            #   disconnects; increase only if Drive I/O
                            #   becomes the bottleneck, which is unlikely here)
_ckpt_suffix     = f"{ARM_SUFFIX}{'_pilot' if PILOT_MODE else ''}"
# RUN_SUFFIX distinguishes ABLATION ARMS within the same model folder.
# Without the refinement config in the filename, a 'grounded' run and an
# 'intrinsic' run would share one checkpoint: the second run would find all
# cases already processed, execute ZERO cases, and silently produce no data —
# while also overwriting the first run's CSV/plots. The Drive folder stays
# named by MODEL_KEY (ARM) so each model keeps its own directory.
_refine_tag = f"{REFINEMENT_FEEDBACK_MODE}-{'guard' if REFINEMENT_ACCEPT_GUARD else 'noguard'}"
RUN_SUFFIX       = f"{_ckpt_suffix}__{_refine_tag}"
CHECKPOINT_LOCAL = f"/content/checkpoint_{RUN_SUFFIX}.jsonl"
CHECKPOINT_DRIVE = os.path.join(DRIVE_ROOT, "results", ARM, f"checkpoint_{RUN_SUFFIX}.jsonl")
print(f"§2  RUN IDENTITY\n"
      f"    MODEL_KEY (candidate) : {MODEL_KEY}\n"
      f"    JUDGE (fixed)         : {JUDGE_KEY}\n"
      f"    PILOT_MODE            : {PILOT_MODE}"
      f"{f' ({PILOT_N_CASES} cases)' if PILOT_MODE else ' (full case bank)'}\n"
      f"    Refinement mode       : {REFINEMENT_FEEDBACK_MODE}\n"
      f"    Acceptance guard      : {REFINEMENT_ACCEPT_GUARD}\n"
      f"    RUN_SUFFIX            : {RUN_SUFFIX}\n"
      f"    Drive folder          : {os.path.join(DRIVE_ROOT, 'results', ARM)}\n"
      f"    Checkpoint            : {CHECKPOINT_LOCAL}\n")

# ── Memory management for free-tier Colab (~12.7 GB system RAM) ──────────────
# True  (default, recommended on free tier): candidate stays resident; the
#        judge is loaded on demand around each batch of reward calls and
#        released immediately after, so candidate+judge are never BOTH fully
#        resident in host RAM at the same time. Costs ~2 extra model loads
#        per case (~10-40s each depending on Drive I/O) in exchange for a
#        much lower peak RAM footprint.
# False (only if you have Colab Pro / a high-RAM runtime): both models load
#        once in §3 and stay resident for the whole run — faster, no reload
#        overhead, but requires enough RAM to hold both simultaneously.
LOW_RAM_MODE = True
EMBEDDING_MODEL_NAME = "intfloat/multilingual-e5-large"
                        # HARMONIZED (was "-small", 384-dim). RAG and GraphRAG
                        # both use e5-large (1024-dim) and share ONE canonical
                        # FAISS index built with it. Running RL on e5-small
                        # meant this arm retrieved from a DIFFERENT vector
                        # store than the arms it is compared against — a
                        # substrate confound, not an architectural difference.
                        # The original reason for "-small" (~470MB vs ~2.24GB)
                        # was VRAM pressure while the judge was GPU-resident;
                        # the judge now runs on CPU (n_gpu_layers=0), which
                        # frees ~2.5GB and makes e5-large affordable.

DECISION_LABELS   = ["Procedente", "Improcedente", "Parcialmente Improcedente"]
# ^ verified against gold_prediction_90_compatible.json: {Procedente: 49,
#   Improcedente: 26, "Parcialmente Improcedente": 15}. UPDATED for the
#   rebalanced 90-case set — the former 60-case distribution (49/10/1) left
#   the majority-class baseline at 81.7%, which no arm beat; it is now 54.4%,
#   and every class has support sufficient for per-class P/R/F1 and CIs.
#   Exact casing matters —
#   DECISION_LABEL2ID does a literal string match against the LLM's
#   `decision_estimada` output (see _STRATEGY_SYSTEM in §8).
DECISION_LABEL2ID = {l: i for i, l in enumerate(DECISION_LABELS)}

print(f"§2  Model={MODEL_KEY} | arch={MODEL_ARCH} {MODEL_PARAMS_B}B | "
      f"quant={MODEL_QUANT} | n_ctx={N_CTX_EFFECTIVE} "
      f"{'(override)' if MODEL_SPEC.get('n_ctx') else '(global default)'} | arm={ARM}")

# ══════════════════════════════════════════════════════════════════════════
# CANONICAL ARTICLE NORMALIZER — single source of truth on Drive
# ══════════════════════════════════════════════════════════════════════════
# Legal article citations appear in incompatible surface forms across the gold
# set and the arms:
#     gold      "LFT arts. 20–21"   "CPEUM art. 123-A"
#     RAG/GRAG  "20 Ley Federal del Trabajo (LFT)"
#     RL        "LFT art. 12"
# Comparing these as raw strings (only .strip().lower()) scores a correct
# citation as a miss purely because of formatting. Measured impact on the
# 60-case draft: article F1 of 0.005 / 0.000 / 0.048 raw, versus 0.134 / 0.159
# / 0.207 canonicalized — i.e. the raw numbers were an artifact of output
# format, not of legal grounding.
#
# norm.py lives ONCE on Drive and is imported by ALL FOUR arms and by
# CrossModelCompar. A per-notebook copy would drift and produce two different
# article scores for the same run with no visible cause. Missing module is a
# HARD FAILURE — a silent fallback to raw matching is exactly the bug this
# module exists to remove.
_LIB_DIR = os.path.join(DRIVE_ROOT, "lib")
if _LIB_DIR not in sys.path:
    sys.path.insert(0, _LIB_DIR)
try:
    from norm import normalize_articles
except ModuleNotFoundError as _e:
    raise ModuleNotFoundError(
        f"Canonical article normalizer not found: {_LIB_DIR}/norm.py\n"
        f"  Upload norm.py to {_LIB_DIR}/ before running. Article grounding "
        f"metrics are invalid without it (raw string matching scores correct "
        f"citations as misses whenever the surface form differs from gold)."
    ) from _e
print(f"  ✅ canonical article normalizer loaded from {_LIB_DIR}/norm.py")
try:
    from index_manifest import ensure_manifest
except ModuleNotFoundError as _e:
    raise ModuleNotFoundError(
        f"index_manifest.py not found in {_LIB_DIR}/ — required to verify that "
        f"this arm retrieves from the same frozen index as the others."
    ) from _e


In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# §PRE  PRE-FLIGHT CHECKS
# ──────────────────────────────────────────────────────────────────────────────
print("\n── §PRE  Pre-flight checks ────────────────────────────────────────────")
_pf_ok = True

def _pf(label, cond, fix=""):
    global _pf_ok
    if cond:
        print(f"  ✅  {label}")
    else:
        print(f"  ❌  {label}")
        if fix: print(f"       Fix: {fix}")
        _pf_ok = False

def _resolve_gguf(pattern, search_dirs):
    pat = pattern.lower()
    hits = []
    for d in search_dirs:
        if os.path.isdir(d):
            for fn in os.listdir(d):
                if fn.lower().endswith(".gguf") and _fnmatch.fnmatch(fn.lower(), pat):
                    hits.append(os.path.join(d, fn))
    return sorted(set(hits))

_SEARCH_DIRS = [MODELS_DIR_LOCAL, MODELS_DIR_DRIVE, "/content", DRIVE_ROOT]

_pf("Drive mounted", os.path.isdir(DRIVE_ROOT),
    "Run drive.mount('/content/drive') or enable Drive in Colab.")

_cand_hits = _resolve_gguf(MODEL_SPEC["pattern"], _SEARCH_DIRS)
_pf(f"Candidate GGUF '{MODEL_SPEC['pattern']}' found", bool(_cand_hits),
    f"Place the GGUF in {MODELS_DIR_DRIVE}/")

_judge_hits = _resolve_gguf(JUDGE_SPEC["pattern"], _SEARCH_DIRS)
_pf(f"Judge GGUF '{JUDGE_SPEC['pattern']}' found", bool(_judge_hits),
    f"Place Phi-3.5-mini-instruct-Q4_K_M.gguf in {MODELS_DIR_DRIVE}/")

_pf("Gold case bank (90 cases) found", os.path.isfile(GOLD_JSON_PATH),
    f"Upload to {EVAL_JSON_PATH}/")
_pf("Rubric JSON (20 questions) found", os.path.isfile(RUBRIC_JSON_PATH),
    f"Upload to {EVAL_JSON_PATH}/")

if _HAS_GPU:
    _vmem = subprocess.run(
        ["nvidia-smi", "--query-gpu=memory.free", "--format=csv,noheader,nounits"],
        capture_output=True, text=True)
    try:
        _free_gb = int(_vmem.stdout.strip().split("\n")[0]) / 1024
        if LOW_RAM_MODE:
            # only the candidate is guaranteed resident continuously; the
            # judge is loaded/released around each reward batch
            _need_gb = MODEL_PARAMS_B * 0.65 + 1.5
        else:
            _need_gb = MODEL_PARAMS_B * 0.65 + JUDGE_SPEC["params_b"] * 0.65 + 1.5
        _pf(f"Free VRAM ≥ {_need_gb:.1f} GB "
            f"(LOW_RAM_MODE={LOW_RAM_MODE}; have {_free_gb:.1f} GB)",
            _free_gb >= _need_gb, "Free VRAM, or set LOW_RAM_MODE=True in §2.")
    except Exception:
        print("  ⚠️  Could not parse VRAM — proceeding.")

try:
    import psutil
    _ram = psutil.virtual_memory()
    _ram_avail_gb, _ram_total_gb = _ram.available / 1e9, _ram.total / 1e9
    # rough estimate: candidate weights + embeddings + Colab/Python baseline;
    # the judge is excluded here when LOW_RAM_MODE=True since it is not held
    # resident at the same time as the candidate under normal operation
    _ram_need_gb = (MODEL_PARAMS_B * 0.9) + 1.0 + 2.0
    if not LOW_RAM_MODE:
        _ram_need_gb += JUDGE_SPEC["params_b"] * 0.9
    print(f"  ℹ️   System RAM: {_ram_avail_gb:.1f} GB free / {_ram_total_gb:.1f} GB total "
          f"— rough estimate needs ~{_ram_need_gb:.1f} GB")
    if _ram_avail_gb < _ram_need_gb:
        print(f"  ⚠️  Free RAM looks tight for this config. LOW_RAM_MODE is "
              f"{'already ON' if LOW_RAM_MODE else 'OFF — turn it ON in §2'}; "
              f"and restarting the runtime to clear leftover memory from a "
              f"previous run before continuing. Do NOT switch to a smaller "
              f"embedding model: the document index is a FROZEN artifact "
              f"shared by all arms and built with e5-large (1024-dim); "
              f"changing the embedder would break the manifest assertion and "
              f"invalidate cross-arm comparability.")
except ImportError:
    print("  ⚠️  psutil not available — skipping the system-RAM estimate.")

# llama_cpp is the single most failure-prone dependency here (CPU-only wheels
# import fine but silently ignore n_gpu_layers=-1; CUDA-mismatched wheels
# install fine but fail to import). Settle both questions in pre-flight
# rather than discovering them mid-benchmark.
try:
    import llama_cpp as _lcpp_probe
    try:
        _gpu_offload = bool(_lcpp_probe.llama_supports_gpu_offload())
    except Exception:
        _gpu_offload = None
    if _gpu_offload is True:
        print("  ✅  llama-cpp-python imports and reports GPU offload support")
    elif _gpu_offload is False:
        print("  ⚠️  llama-cpp-python is a CPU-ONLY build — n_gpu_layers=-1 "
              "will be ignored and generation will be MUCH slower "
              "(expect ~an hour per case rather than minutes). Re-run §0 "
              "after deleting the sentinel to retry a CUDA wheel.")
    else:
        print("  ℹ️   llama-cpp-python imports; GPU offload status unknown")
except Exception as e:
    _pf(f"llama-cpp-python imports ({type(e).__name__}: {e})", False,
        "Re-run §0 (delete the sentinel in /tmp to force reinstall).")

# VRAM budget check. Once GPU offload actually works, the binding constraint
# shifts from host RAM to VRAM, and the KV cache — not the weights — is
# usually what overflows. Phi-3.5-mini has no GQA, so its cache costs ~3x
# per token what Llama-3.1-8B's does despite being half the size.
if _HAS_GPU:
    def _kv_gb(layers, kv_heads, head_dim, n_ctx):
        return 2 * layers * kv_heads * head_dim * n_ctx * 2 / 1e9

    _cand_gb = MODEL_PARAMS_B * 0.62 + _kv_gb(32, 8, 128, N_CTX)
    _judge_gb = JUDGE_SPEC["params_b"] * 0.62 + _kv_gb(32, 32, 96, JUDGE_SPEC["n_ctx"])
    _budget_gb = _cand_gb + _judge_gb
    try:
        _tot = subprocess.run(
            ["nvidia-smi", "--query-gpu=memory.total", "--format=csv,noheader,nounits"],
            capture_output=True, text=True)
        _total_gb = int(_tot.stdout.strip().split("\n")[0]) / 1024
    except Exception:
        _total_gb = 15.0
    print(f"  ℹ️   VRAM budget: candidate ~{_cand_gb:.1f} GB (n_ctx={N_CTX_EFFECTIVE}) + "
          f"judge ~{_judge_gb:.1f} GB (n_ctx={JUDGE_SPEC['n_ctx']}) "
          f"= ~{_budget_gb:.1f} GB of {_total_gb:.1f} GB")
    if _budget_gb > _total_gb * 0.95:
        print(f"  ⚠️  Estimated VRAM need (~{_budget_gb:.1f} GB) is close to or "
              f"over capacity — expect 'Failed to create llama_context'. "
              f"Lower JUDGE_SPEC['n_ctx'] or N_CTX in §2. (LazyLLM will retry "
              f"at smaller n_ctx automatically, but scores may be affected "
              f"by prompt truncation.)")

if not _pf_ok:
    raise RuntimeError("❌  Pre-flight failed. Fix the issues above before continuing.")
print("✅  All pre-flight checks passed.\n")


def print_ram_checkpoint(label: str) -> None:
    """Cheap diagnostic used across §3/§5/§6/§6b to see exactly where host
    RAM climbs — call after any step suspected of holding large objects."""
    try:
        import psutil
        m = psutil.virtual_memory()
        print(f"  🧭  RAM checkpoint [{label}]: {m.used/1e9:.2f} GB used / "
              f"{m.total/1e9:.2f} GB total ({m.percent:.0f}%)")
    except ImportError:
        pass


In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# §3  EMBEDDINGS + JUDGE MANAGER INIT — candidate load DEFERRED to §6c;
#     judge lazy-loaded under LOW_RAM_MODE (see §2) to cap peak host RAM
# ──────────────────────────────────────────────────────────────────────────────
import gc

def _load_gguf(model_key, spec, search_dirs, global_n_ctx, n_ctx_override=None):
    matches = _resolve_gguf(spec["pattern"], search_dirs)
    if not matches:
        raise FileNotFoundError(f"No GGUF matching '{spec['pattern']}' for '{model_key}'.")
    resolved = matches[0]
    local_path = os.path.join("/content", os.path.basename(resolved))
    if resolved.startswith("/content"):
        model_path = resolved
    elif os.path.exists(local_path):
        model_path = local_path
    else:
        try:
            _shutil_model.copy2(resolved, local_path)
            model_path = local_path
        except Exception as e:
            # Not silent: reading a multi-GB GGUF directly off mounted Drive
            # is far slower than local disk and would look like unexplained
            # slow inference rather than an I/O problem.
            print(f"  ⚠️  Could not copy model to local disk ({type(e).__name__}: {e}). "
                  f"Loading directly from Drive — this is significantly slower.")
            model_path = resolved
    n_ctx = n_ctx_override or spec.get("n_ctx") or global_n_ctx
    print(f"    → '{model_key}' loading with n_ctx={n_ctx} "
          f"(override={n_ctx_override}, registry={spec.get('n_ctx')}, "
          f"global_default={global_n_ctx}) — this is the value actually "
          f"passed to Llama().")
    # FIX: _load_gguf serves BOTH the candidate and the judge. Read the
    # offload policy from the spec (default -1 = full GPU) so JUDGE_SPEC can
    # pin the judge to CPU without affecting the candidate.
    _ngl = spec.get("n_gpu_layers", -1)
    kwargs = dict(model_path=model_path, n_ctx=n_ctx, n_gpu_layers=_ngl, verbose=False,
                 use_mmap=False,   # see §2 note: avoids page-cache RSS spikes that
                                   # can trigger a cgroup OOM kill on free-tier
                                   # Colab containers during/after full GPU offload
                 use_mlock=False)  # never pin pages — let the OS reclaim freely
    if spec.get("chat_format"):
        kwargs["chat_format"] = spec["chat_format"]
    print(f"  Loading '{model_key}' from {model_path} (n_ctx={n_ctx}, use_mmap=False)...")
    return Llama(**kwargs)


class LazyLLM:
    """Loads a GGUF model on first use and can release it on demand. Used for
    the judge under LOW_RAM_MODE so the candidate and judge are never both
    fully resident at the same time — trades ~2 extra loads per case for a
    lower peak footprint (see §2).

    get() retries with progressively smaller n_ctx if the model cannot be
    loaded. Without this, a VRAM shortage raises before `self._llm` is ever
    assigned, so `_llm` stays None and EVERY subsequent call retries the
    identical failing load — producing an endless 'Loading...' / 'Failed to
    create llama_context' loop that silently scores every rubric group as
    0 compliance."""
    def __init__(self, model_key, spec, search_dirs, default_n_ctx):
        self.model_key, self.spec = model_key, spec
        self.search_dirs, self.default_n_ctx = search_dirs, default_n_ctx
        self._llm = None
        self._n_ctx_in_use = None

    def get(self):
        if self._llm is not None:
            return self._llm
        base = self._n_ctx_in_use or self.spec.get("n_ctx") or self.default_n_ctx
        ladder = sorted({n for n in (base, 4096, 3072, 2048) if n <= base},
                        reverse=True) or [base]
        last_err = None
        for n_ctx in ladder:
            try:
                self._llm = _load_gguf(self.model_key, self.spec, self.search_dirs,
                                       self.default_n_ctx, n_ctx_override=n_ctx)
                if n_ctx != base:
                    print(f"  ⚠️  '{self.model_key}' loaded at REDUCED n_ctx={n_ctx} "
                          f"(requested {base}) — VRAM was insufficient. Long "
                          f"judge prompts may be truncated; scores from this "
                          f"run should be treated with caution.")
                self._n_ctx_in_use = n_ctx
                return self._llm
            except Exception as e:
                last_err = e
                print(f"  ⚠️  '{self.model_key}' failed to load at n_ctx={n_ctx} "
                      f"({type(e).__name__}: {e})"
                      + (" — retrying smaller" if n_ctx != ladder[-1] else ""))
                gc.collect()
        raise RuntimeError(
            f"Could not load '{self.model_key}' at any n_ctx in {ladder}. "
            f"Last error: {last_err}. Free VRAM (e.g. lower N_CTX for the "
            f"candidate in §2) and retry.")

    def release(self):
        if self._llm is not None:
            del self._llm
            self._llm = None
            gc.collect()
            if _HAS_GPU:
                try:
                    import torch
                    torch.cuda.empty_cache()
                except Exception:
                    pass

    @property
    def is_loaded(self):
        return self._llm is not None


print("§3  Preparing model managers (candidate load DEFERRED to §6c — see note)...")
# The candidate GGUF (4.5-9 GB) is NOT loaded here anymore. It isn't needed
# until §7/§9, so loading it now would sit resident in RAM through the
# heaviest phase of the whole pipeline: PDF ingestion + embedding the full
# SCJN + labor-law corpus (§5/§6). That combination is what caused the OOM
# kernel restart. Deferring the load to §6c (right after the FAISS indexes
# are built) removes the candidate's footprint from that peak entirely.

judge_mgr = LazyLLM(JUDGE_KEY, JUDGE_SPEC, _SEARCH_DIRS, JUDGE_SPEC["n_ctx"])
if LOW_RAM_MODE:
    print(f"  LOW_RAM_MODE=True — judge '{JUDGE_KEY}' will load on demand "
          f"around each reward batch, not now.")
else:
    print(f"  LOW_RAM_MODE=False — loading judge '{JUDGE_KEY}' eagerly "
          f"(co-resident with the candidate for the whole run)...")
    judge_mgr.get()
    print_ram_checkpoint("after judge load")

# CORRECTIVE FIX · embedder pinned to CPU in this arm. The device budget was
# allocated backwards for this workload: the embedder held ~2.2 GB of VRAM to
# serve ~90 retrieval queries plus one CBR index build, while the judge — called
# ~2,700 times per run — was displaced to CPU. Swapping them costs a negligible
# amount of embedding time and returns the VRAM to the component that dominates
# runtime. Retrieval results are unaffected: the same model produces the same
# vectors on either device, and the frozen-index assertion still verifies
# dimensionality against the manifest.
embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL_NAME,
    model_kwargs={"device": "cpu"})
_EMB_DIM = len(embeddings.embed_query("dimension probe"))
print(f"  Embedding model '{EMBEDDING_MODEL_NAME}' → vector dim={_EMB_DIM}")
print_ram_checkpoint("after embeddings load")
print(f"✅  §3 complete (LOW_RAM_MODE={LOW_RAM_MODE}, "
      f"embeddings={EMBEDDING_MODEL_NAME}). Candidate load deferred to §6c.\n")


In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# §4  HELPER FUNCTIONS
# ──────────────────────────────────────────────────────────────────────────────
def load_pdfs_from_directory(directory_path: str) -> list:
    docs = []
    if not os.path.isdir(directory_path):
        print(f"  ⚠️  Not found: {directory_path}"); return docs
    for fn in sorted(os.listdir(directory_path)):
        if fn.lower().endswith(".pdf"):
            try:
                docs.extend(PyPDFLoader(os.path.join(directory_path, fn)).load())
            except Exception as e:
                print(f"    ❌  {fn}: {e}")
    print(f"  {len(docs)} pages from '{directory_path}'.")
    return docs


def sanitize_text(text: str, max_chars: int = 1_400_000) -> str:
    if not text or not isinstance(text, str):
        return ""
    try:
        text = text.encode("utf-16", "surrogatepass").decode("utf-16", "ignore")
        text = text.encode("utf-8", "ignore").decode("utf-8", "ignore")
    except Exception:
        pass
    return text[:max_chars].strip()


def load_case_bank(json_path: str) -> list:
    """Parses the 60-case gold bank into CBR exemplars (facts + gold outcome)."""
    if not os.path.isfile(json_path):
        return []
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    out = []
    for c in data.get("cases", []):
        gs, cel = c.get("gold_standard", {}), c.get("expected_cel_output", {})
        out.append({
            "case_id": c.get("case_id", "UNKNOWN"),
            "facts": sanitize_text((c.get("input_summary", "") + "\n\n" + c.get("facts", "")).strip()),
            "gold_decision": gs.get("decision", ""),
            "gold_articles": cel.get("articulos_citados", []),
            "conclusiones": cel.get("conclusiones_legales", []),
            "category": c.get("benchmark_metadata", {}).get("category", ""),
        })
    print(f"  Loaded {len(out)} cases from case bank '{json_path}'.")
    return out


def load_rubric(json_path: str) -> list:
    if not os.path.isfile(json_path):
        return []
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    qs = data.get("preguntas") or data.get("questions") or []
    print(f"  Loaded {len(qs)} rubric questions from '{json_path}'.")
    return qs


def _flatten_articles(x):
    if isinstance(x, list):
        out = []
        for i in x:
            out.extend(_flatten_articles(i))
        return out
    return [str(x)] if x else []


def build_messages(system_prompt: str, user_prompt: str, supports_system_role: bool):
    if supports_system_role:
        return [{"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}]
    return [{"role": "user", "content": f"{system_prompt}\n\n{user_prompt}"}]


def chat(llm_obj, system_prompt, user_prompt, supports_system_role=True,
         temperature=0.4, top_p=0.9, max_tokens=900, repeat_penalty=1.0):
    msgs = build_messages(system_prompt, user_prompt, supports_system_role)
    resp = llm_obj.create_chat_completion(
        messages=msgs, temperature=temperature, top_p=top_p, max_tokens=max_tokens,
        repeat_penalty=repeat_penalty)
    text = resp["choices"][0]["message"]["content"]
    usage = resp.get("usage", {})
    return text, usage


def safe_json_extract(text: str) -> dict:
    """Extracts a JSON object from an LLM response, tolerating the three ways
    small quantized models routinely break strict parsing:
      1. markdown code fences around the JSON
      2. trailing prose after the closing brace
      3. TRUNCATION mid-structure (hit max_tokens before closing brackets)

    The previous implementation was `re.search(r'\\{.*\\}', ..., DOTALL)` +
    json.loads, which failed completely on (3): a judge response listing
    questions 9-12 that got cut off inside question 12 was discarded whole,
    throwing away the three questions that HAD been scored correctly and
    silently zeroing the entire group. This version walks the string tracking
    brace depth (string-aware, escape-aware), returns the first balanced
    object if there is one, and otherwise salvages every complete element
    before the truncation point and closes the open brackets."""
    if not text or not isinstance(text, str):
        return {}
    t = re.sub(r"```(?:json)?", "", text).strip()
    start = t.find("{")
    if start == -1:
        return {}

    depth, in_str, esc, stack, last_safe = 0, False, False, [], None
    for i in range(start, len(t)):
        ch = t[i]
        if esc:
            esc = False
            continue
        if ch == "\\":
            esc = True
            continue
        if ch == '"':
            in_str = not in_str
            continue
        if in_str:
            continue
        if ch in "{[":
            stack.append(ch)
            depth += 1
        elif ch in "}]":
            if stack:
                stack.pop()
            depth -= 1
            if depth == 0:
                try:
                    return json.loads(t[start:i + 1])
                except Exception:
                    break
            # depth 2 == just closed one element inside {"results": [ ... ]}
            if depth == 2:
                last_safe = i

    if last_safe is not None:
        frag = t[start:last_safe + 1]
        d_str, d_esc, open_stack = False, False, []
        for ch in frag:
            if d_esc:
                d_esc = False
                continue
            if ch == "\\":
                d_esc = True
                continue
            if ch == '"':
                d_str = not d_str
                continue
            if d_str:
                continue
            if ch in "{[":
                open_stack.append(ch)
            elif ch in "}]":
                if open_stack:
                    open_stack.pop()
        closers = "".join("}" if c == "{" else "]" for c in reversed(open_stack))
        for cand in (frag + closers, frag.rstrip().rstrip(",") + closers):
            try:
                return json.loads(cand)
            except Exception:
                continue
    return {}


def load_checkpoint(local_path: str, drive_path: str) -> tuple:
    """Resume support for §9. Prefers the Drive copy (survives a Colab
    disconnection even if /content was wiped); falls back to the local copy.
    Returns (rows, processed_case_ids)."""
    src = drive_path if os.path.exists(drive_path) else (
        local_path if os.path.exists(local_path) else None)
    rows = []
    if src:
        with open(src, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                try:
                    rows.append(json.loads(line))
                except json.JSONDecodeError:
                    continue  # tolerate a truncated last line from a mid-write disconnect
        if src == drive_path and not os.path.exists(local_path):
            _shutil_model.copy2(drive_path, local_path)
        print(f"  ↻  Resuming from checkpoint: {len(rows)} case(s) already processed "
              f"({src}).")
    return rows, {r["case_id"] for r in rows}


def append_checkpoint(row: dict, local_path: str, drive_path: str) -> None:
    """Appends one result row (JSONL) and mirrors it to Drive immediately, so
    a disconnection loses at most CHECKPOINT_EVERY cases, never the whole run."""
    with open(local_path, "a", encoding="utf-8") as f:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")
    os.makedirs(os.path.dirname(drive_path), exist_ok=True)
    _shutil_model.copy2(local_path, drive_path)

def _embedding_meta_path(index_dir: str) -> str:
    return os.path.join(index_dir, "_embedding_meta.json")


def write_embedding_meta(index_dir: str) -> None:
    """Tags a FAISS index directory with the embedding model that built it.
    Loading a FAISS index built with a different embedding model (different
    vector dimensionality) doesn't fail at load time — it fails later, inside
    faiss's C++ layer, as a bare `AssertionError: assert d == self.d` with no
    message, the first time a query is run against it. Tagging + validating
    the cache prevents that failure mode entirely."""
    try:
        with open(_embedding_meta_path(index_dir), "w", encoding="utf-8") as f:
            json.dump({"model": EMBEDDING_MODEL_NAME, "dim": _EMB_DIM}, f)
    except Exception as e:
        print(f"  ⚠️  Could not write embedding metadata for {index_dir}: {e}")


def embedding_cache_matches(index_dir: str) -> bool:
    """True only if index_dir carries a metadata sidecar matching the
    CURRENT EMBEDDING_MODEL_NAME. Missing or mismatched metadata (e.g. an
    index built before this check existed, or built with a different
    embedding model) means the cache is stale and must be rebuilt, never
    loaded as-is."""
    p = _embedding_meta_path(index_dir)
    if not os.path.isfile(p):
        return False
    try:
        with open(p, encoding="utf-8") as f:
            meta = json.load(f)
        return meta.get("model") == EMBEDDING_MODEL_NAME
    except Exception:
        return False


print("✅  §4 helpers ready.\n")


In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# §5-§6  PDF INGESTION + RAG VECTOR INDEX — streamed, one file at a time
# ──────────────────────────────────────────────────────────────────────────────
# Previously: load every PDF into one big list, split ALL of it, then build
# the FAISS index from the full combined chunk list — meaning the entire
# corpus sat resident in RAM at once. On the full SCJN + labor-law corpus
# that was almost certainly the proximate cause of the OOM kernel restart in
# §5. Streaming (load -> split -> add-to-index -> discard, per file) caps the
# peak to roughly one file's pages/chunks, independent of corpus size.
print("── §5-§6  PDF Ingestion + RAG Vector Index (streamed) ─────────────────")
_faiss_local = "/content/faiss_index"
# ── CORRECTIVE FIX · document-index cache validity ───────────────────────────
# The legacy guard below required a sidecar file (_embedding_meta.json) written
# INSIDE the index directory by THIS notebook. The canonical document index is
# built and owned by the GraphRAG arm, which does not write that sidecar — so
# the guard classified a perfectly valid, manifest-certified index as "stale"
# and triggered a full rebuild from the PDFs.
#
# That rebuild is destructive: it would overwrite, at FAISS_INDEX_PATH, the very
# index against which the already-completed bare-LLM, RAG and GraphRAG
# conditions were run, silently breaking cross-arm comparability — the exact
# failure the frozen manifest exists to prevent.
#
# The manifest (asserted below, after load) is strictly stronger than the
# sidecar: it verifies embedder, dimensionality, vector count AND chunk
# parameters, and hard-fails on divergence instead of rebuilding. The sidecar
# check is therefore redundant for the SHARED document index and is dropped
# here. It is retained for the CBR index, which this arm exclusively owns and
# legitimately rebuilds from the 90-case bank.
_faiss_cached = (os.path.exists(_faiss_local) or os.path.exists(FAISS_INDEX_PATH))

def _iter_pdf_paths(directory: str):
    if not os.path.isdir(directory):
        print(f"  ⚠️  Not found: {directory}")
        return
    for fn in sorted(os.listdir(directory)):
        if fn.lower().endswith(".pdf"):
            yield os.path.join(directory, fn)


def stream_build_faiss_index(sources: list, embeddings, gc_every: int = 5):
    """sources: list of (directory, chunk_size, chunk_overlap). Builds a
    FAISS index incrementally, one PDF at a time, freeing each file's pages
    and chunks before moving to the next."""
    seps = ["\n\n", "\n", ".", " ", ""]
    vector_db, n_files, n_chunks = None, 0, 0
    for directory, chunk_size, chunk_overlap in sources:
        splitter = RecursiveCharacterTextSplitter(
            chunk_size=chunk_size, chunk_overlap=chunk_overlap, separators=seps)
        for path in _iter_pdf_paths(directory):
            try:
                pages = PyPDFLoader(path).load()
                chunks = splitter.split_documents(pages)
            except Exception as e:
                print(f"    ❌  {os.path.basename(path)}: {type(e).__name__}: {e}")
                continue
            if chunks:
                if vector_db is None:
                    vector_db = FAISS.from_documents(chunks, embeddings)
                else:
                    vector_db.add_documents(chunks)
                n_chunks += len(chunks)
            n_files += 1
            del pages, chunks
            if n_files % gc_every == 0:
                gc.collect()
                print(f"    ...{n_files} file(s), {n_chunks} chunk(s) indexed so far")
    if vector_db is None:
        raise RuntimeError("No chunks produced — check PDF paths (RAW_PDFS_PATH/LAW_PDFS_PATH).")
    print(f"  {n_files} PDF(s) → {n_chunks} chunk(s) total")
    return vector_db


if _faiss_cached:
    print("  ℹ️   Cached FAISS index found and matches the current embedding "
          f"model ('{EMBEDDING_MODEL_NAME}') — skipping PDF ingestion for "
          "this arm (saves RAM/time on arms 2-5 of the benchmark).")
    if os.path.exists(_faiss_local):
        vector_db = FAISS.load_local(_faiss_local, embeddings, allow_dangerous_deserialization=True)
    else:
        vector_db = FAISS.load_local(FAISS_INDEX_PATH, embeddings, allow_dangerous_deserialization=True)
        vector_db.save_local(_faiss_local)
        write_embedding_meta(_faiss_local)
else:
    if os.path.exists(_faiss_local) or os.path.exists(FAISS_INDEX_PATH):
        print("  ♻️  A cached FAISS index exists but was built with a "
              "different embedding model (or predates this check) — "
              "ignoring it and rebuilding from the PDFs to avoid a "
              "dimension-mismatch crash in FAISS.")
    raw_src = RAW_PDFS_PATH if os.path.exists(RAW_PDFS_PATH) else LOCAL_RAW_PDFS
    law_src = LAW_PDFS_PATH if os.path.exists(LAW_PDFS_PATH) else LOCAL_LAW_PDFS
    vector_db = stream_build_faiss_index(
        [(raw_src, CHUNK_SIZE_PREC, CHUNK_OVERLAP_PREC),
         (law_src, CHUNK_SIZE_LAW, CHUNK_OVERLAP_LAW)],
        embeddings)
    vector_db.save_local(_faiss_local)
    write_embedding_meta(_faiss_local)

# FIX 3 · frozen-index assertion (hard-fail). Verifies embedder, dim, ntotal AND
# chunking against the manifest — a dim-only check cannot distinguish two
# 1024-dim indexes built from different chunkings.
ensure_manifest(FAISS_INDEX_PATH, vector_db, embedder=EMBEDDING_MODEL_NAME,
                chunk_prec=CHUNK_SIZE_PREC, overlap_prec=CHUNK_OVERLAP_PREC,
                chunk_law=CHUNK_SIZE_LAW,   overlap_law=CHUNK_OVERLAP_LAW,
                arm=ARM, embed_fn=embeddings.embed_query)

retriever = vector_db.as_retriever(search_kwargs={"k": TOP_K_RAG})
gc.collect()
print(f"  ✅  {vector_db.index.ntotal:,} document vectors ready "
      f"(dim={_EMB_DIM}, model={EMBEDDING_MODEL_NAME}).\n")
print_ram_checkpoint("after RAG index build (streamed)")


# ──────────────────────────────────────────────────────────────────────────────
# §6b  CBR VECTOR INDEX (FAISS, 90-case exemplar bank) — shared across all arms
# ──────────────────────────────────────────────────────────────────────────────
print("── §6b  CBR Vector Index (case bank) ──────────────────────────────────")
case_bank = load_case_bank(GOLD_JSON_PATH)
_cbr_local = "/content/faiss_index_cbr"

if case_bank:
    if os.path.exists(_cbr_local) and embedding_cache_matches(_cbr_local):
        cbr_db = FAISS.load_local(_cbr_local, embeddings, allow_dangerous_deserialization=True)
    elif os.path.exists(CBR_INDEX_PATH) and embedding_cache_matches(CBR_INDEX_PATH):
        cbr_db = FAISS.load_local(CBR_INDEX_PATH, embeddings, allow_dangerous_deserialization=True)
        cbr_db.save_local(_cbr_local)
        write_embedding_meta(_cbr_local)
    else:
        if os.path.exists(_cbr_local) or os.path.exists(CBR_INDEX_PATH):
            print("  ♻️  Ignoring a stale CBR index (different embedding "
                  "model) — rebuilding from the 60-case bank.")
        from langchain_core.documents import Document as _LDoc
        cbr_docs = [_LDoc(page_content=c["facts"], metadata=c) for c in case_bank if c["facts"]]
        cbr_db = FAISS.from_documents(cbr_docs, embeddings)
        cbr_db.save_local(_cbr_local)
        write_embedding_meta(_cbr_local)
    print(f"  ✅  {cbr_db.index.ntotal:,} case-bank vectors ready "
          f"(CBR retrieval, top_k={TOP_K_CBR}, dim={_EMB_DIM}).\n")
else:
    cbr_db = None
    print("  ⚠️  No case bank loaded — CBR component of the reward will be skipped.\n")
print_ram_checkpoint("after CBR index build")


def retrieve_analogous_cases(facts: str, current_case_id: str = None,
                             k: int = TOP_K_CBR) -> list:
    """CORRECTIVE FIX (post-hoc methodological review): the CBR exemplar bank
    is built from the SAME 60-case gold set used for evaluation
    (gold_prediction_90_compatible.json — see §2 GOLD_JSON_PATH), so a naive
    similarity_search over facts nearly-identical to the query returns the
    query's OWN case (cosine similarity ~1.0) as its most "analogous" match.
    That leaked the gold_decision/gold_articles of the very case under
    evaluation directly into the generation prompt (_build_context) AND into
    the reward signal (cbr_outcome_alignment) — a severe train/test
    contamination that invalidates any advantage of this arm over RAG/
    GraphRAG until fixed. We over-fetch by 1 and drop any hit whose
    case_id matches the case currently being solved."""
    if cbr_db is None:
        return []
    hits = cbr_db.similarity_search(facts, k=k + 1 if current_case_id else k)
    metas = [h.metadata for h in hits]
    if current_case_id:
        metas = [m for m in metas if m.get("case_id") != current_case_id]
    return metas[:k]


# ──────────────────────────────────────────────────────────────────────────────
# §6c  CANDIDATE MODEL LOAD (deferred from §3 — see rationale there)
# ──────────────────────────────────────────────────────────────────────────────
def _gpu_mem_used_mb() -> int:
    try:
        r = subprocess.run(
            ["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"],
            capture_output=True, text=True)
        return int(r.stdout.strip().split("\n")[0])
    except Exception:
        return -1


print(f"── §6c  Loading candidate '{MODEL_KEY}' (now that PDF ingestion + "
      f"indexing are done) ──")
_vram_before = _gpu_mem_used_mb() if _HAS_GPU else -1
llm = _load_gguf(MODEL_KEY, MODEL_SPEC, _SEARCH_DIRS, N_CTX)
gc.collect()
if _HAS_GPU:
    _vram_after = _gpu_mem_used_mb()
    _vram_delta = _vram_after - _vram_before if (_vram_before >= 0 and _vram_after >= 0) else -1
    if _vram_delta >= 0:
        print(f"  GPU memory used: {_vram_before} MB → {_vram_after} MB "
              f"(Δ {_vram_delta} MB)")
    if 0 <= _vram_delta < 500:
        print("  ❌  VRAM barely changed after loading an "
              f"{MODEL_PARAMS_B}B model — it almost certainly did NOT load "
              "onto the GPU (n_gpu_layers=-1 is a no-op on a CPU-only "
              "llama-cpp-python build). Check the §0 install log above for "
              "'CUDA build FAILED'. Generation will be extremely slow "
              "(minutes per case became ~an hour) until this is fixed.")
print_ram_checkpoint("after candidate load (deferred)")
print(f"✅  §6c complete: candidate resident for the remainder of the run.\n")


In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# §7  REWARD MODEL — rubric judge (FIXED) + article grounding + CBR alignment
# ──────────────────────────────────────────────────────────────────────────────
print("── §7  Reward Model ────────────────────────────────────────────────────")
rubric_questions = load_rubric(RUBRIC_JSON_PATH)

_JUDGE_SYSTEM = (
    "Eres un juez experto e IMPARCIAL en derecho laboral mexicano. Evalúas "
    "estrategias de litigio SIN generarlas tú mismo. Respondes ÚNICAMENTE en "
    "JSON válido, sin texto adicional."
)


def _serialize_rubric_for_judge(questions: list, include_guideline: bool) -> list:
    """Mirrors the real schema found in rubrica_validacion_contratos.json:
    each question carries 2-4 weighted criteria (57 total across 20
    questions), NOT a single binary cumple/no-cumple. Each criterion is
    tagged with a stable 0-based `idx` so the judge only needs to return a
    positional true/false array — NOT echo back the criterion's `name`
    string. (An earlier version asked the judge to repeat the exact `name`
    text and matched on that string; small deviations in accents/spacing/
    capitalization in the judge's echo silently broke every match and
    produced a flat 0.0% rubric score regardless of actual quality.)
    include_guideline=False strips the (verbose) `guideline` text — used for
    the cheap reward-shaping call inside §7; include_guideline=True is
    reserved for the one-per-case formal evaluation in §11b."""
    payload = []
    for q in questions:
        qid = q.get("id", q.get("question_id", "?"))
        qtext = q.get("question", q.get("pregunta", ""))
        raw_criteria = q.get("criteria") or [{"name": "cumplimiento_general", "weight_pct": 100.0}]
        crits = []
        for idx, c in enumerate(raw_criteria):
            entry = {"idx": idx, "name": c.get("name", ""),
                     "weight_pct": c.get("weight_pct", 100.0 / len(raw_criteria))}
            if include_guideline and c.get("guideline"):
                entry["guideline"] = c["guideline"]
            crits.append(entry)
        item = {"id": qid, "question": qtext, "criteria": crits}
        if q.get("expected_answer"):
            item["expected_answer"] = q["expected_answer"]
        payload.append(item)
    return payload


_JUDGE_FACTS_MAX_CHARS = 6000      # ~1.5K tokens
_JUDGE_STRATEGY_MAX_CHARS = 6000   # ~1.5K tokens
# Together with a rubric group (~1.4K tok with guidelines) and 900 output
# tokens, this keeps every judge prompt inside JUDGE_SPEC["n_ctx"]=6144 even
# for the longest cases in the bank.

def _clip(text: str, max_chars: int) -> str:
    text = text or ""
    return text if len(text) <= max_chars else text[:max_chars] + "\n[...truncado...]"


def _judge_score_group(group_payload: list, strategy_text: str, facts: str,
                       max_tokens: int) -> dict:
    """Scores ONE small group of questions in a single judge call. Splitting
    into groups (instead of all 20 questions/57 criteria in one mega-call)
    is a deliberate reliability tradeoff: the first version asked a 3.8B
    Q4 judge to structure the ENTIRE rubric in one shot, which degenerated
    into a runaway `true, true, true, ...` repetition loop that never closed
    its JSON and got truncated at max_tokens on every call. Small groups are
    a much more tractable structured-output task for a model this size."""
    n_criteria = sum(len(q["criteria"]) for q in group_payload)
    prompt = (
        f"Hechos del caso:\n{_clip(facts, _JUDGE_FACTS_MAX_CHARS)}\n\n"
        f"Estrategia de litigio propuesta:\n"
        f"{_clip(strategy_text, _JUDGE_STRATEGY_MAX_CHARS)}\n\n"
        f"Rúbrica a evaluar (JSON, {len(group_payload)} pregunta(s), "
        f"{n_criteria} criterio(s) en total, cada criterio tiene un índice "
        f"'idx' 0-based dentro de su pregunta):\n"
        f"{json.dumps(group_payload, ensure_ascii=False)}\n\n"
        "Para CADA pregunta, evalúa sus criterios EN ORDEN por su 'idx' y "
        "responde ÚNICAMENTE en JSON con este formato exacto — un arreglo "
        "de booleanos por pregunta, en el mismo orden que los criterios "
        "recibidos (NO repitas el texto de 'name'):\n"
        '{"results": [{"id": <int>, "cumple": [true|false, ...]}, ...]}\n'
        "El arreglo 'cumple' debe tener EXACTAMENTE el mismo número de "
        f"elementos que criterios tiene esa pregunta, incluyendo las "
        f"{len(group_payload)} pregunta(s) recibidas y NADA más."
    )
    raw, parsed = "", {}
    _ids = [q["id"] for q in group_payload]
    try:
        raw, _ = chat(judge_mgr.get(), _JUDGE_SYSTEM, prompt,
                     supports_system_role=JUDGE_SPEC["supports_system_role"],
                     temperature=0.0, max_tokens=max_tokens, repeat_penalty=1.15)
        parsed = safe_json_extract(raw)
    except Exception as e:
        print(f"      ⚠️  judge group call failed [{type(e).__name__}]: {e}")
    # CORRECTIVE FIX (post-hoc review after 1st full run on mistral-7b-v03-q4):
    # the judge (Phi-3.5-mini, 3.8B, Q4) does NOT reliably echo back the real
    # 'id' values it was given — it frequently re-numbers its own 'results'
    # array positionally from 0, regardless of the ids in the prompt. Keying
    # by echoed 'id' silently zeroed whole groups whenever no id happened to
    # collide with a 0-based position (e.g. group [17,18,19,20] -> 0 matches
    # -> entire group lost, vs. group [1,2,3,4] -> partial accidental match).
    # FIX: trust the judge's OUTPUT ORDER (LLMs preserve list order far more
    # reliably than they preserve arbitrary numeric labels) and map each
    # position back to the id we ORIGINALLY sent at that position in
    # group_payload, ignoring whatever 'id' the judge echoed.
    _raw_results = parsed.get("results", []) if parsed else []
    out = {}
    for _pos, _r in enumerate(_raw_results):
        if _pos < len(group_payload):
            out[group_payload[_pos]["id"]] = _r.get("cumple", [])
    _missing = [i for i in _ids if i not in out]
    if not out:
        print(f"      ⚠️  judge group {_ids}: no usable 'results' — scoring 0 "
              f"for this group. Response was {len(raw)} chars; "
              f"head={raw[:160]!r} tail={raw[-160:]!r}")
    elif _missing:
        # Partial recovery is now expected behaviour on truncation rather than
        # total loss: report exactly what survived so the impact is visible.
        print(f"      ℹ️   judge group {_ids}: recovered {sorted(out)}, "
              f"missing {_missing} (likely truncated at {len(raw)} chars / "
              f"max_tokens={max_tokens}) — missing ones score 0.")
    return out


def rubric_batched_judge(strategy_text: str, facts: str, questions: list,
                         include_guideline: bool = False, max_tokens: int = 900,
                         group_size: int = 4) -> dict:
    """Scores a litigation strategy against the FULL weighted rubric via
    SEVERAL small-group calls to the fixed judge model (never the
    candidate) — group_size questions per call, e.g. 20 questions / 4 per
    group = 5 calls, instead of either one call per criterion (~57 calls,
    too slow) or one mega-call for everything (unreliable for a small
    quantized judge — see _judge_score_group). Returns per-question weighted
    scores (0-100 each, summing criterion weight_pct for complying criteria)
    and an overall score_pct = mean of the 20 question scores, matching the
    rubric's own metadata.scoring definition."""
    if not questions:
        return {"score_pct": 50.0, "per_question": []}
    rubric_payload = _serialize_rubric_for_judge(questions, include_guideline)

    results_by_id = {}
    for i in range(0, len(rubric_payload), group_size):
        group = rubric_payload[i:i + group_size]
        results_by_id.update(_judge_score_group(group, strategy_text, facts, max_tokens))

    per_q = []
    for q in rubric_payload:
        cumple_list = results_by_id.get(q["id"], [])
        crit_out, q_score = [], 0.0
        for c in q["criteria"]:
            # a criterion missing/short-indexed in the judge's response is
            # scored conservatively as non-compliant, not silently dropped
            cumple = bool(cumple_list[c["idx"]]) if c["idx"] < len(cumple_list) else False
            q_score += c["weight_pct"] if cumple else 0.0
            crit_out.append({"name": c["name"], "weight_pct": c["weight_pct"], "cumple": cumple})
        per_q.append({"question_id": q["id"], "score_pct": round(q_score, 2), "criteria": crit_out})

    score_pct = sum(p["score_pct"] for p in per_q) / len(per_q) if per_q else 50.0
    return {"score_pct": score_pct, "per_question": per_q}


def article_grounding_f1(pred_articles: list, gold_articles: list) -> float:
    # CORRECTIVE FIX: this component carries 25% of the composite reward
    # (REWARD_WEIGHTS["article_grounding"]). Raw set intersection scored a
    # rollout citing exactly the right articles in a different surface form as
    # 0.0, so best-of-N selection was optimizing partly on FORMATTING rather
    # than legal grounding. Both sides are now canonicalized through the shared
    # normalizer, so "LFT art. 47" and "47 Ley Federal del Trabajo" match.
    pred = normalize_articles(list(map(str, pred_articles)))
    gold = normalize_articles(list(map(str, gold_articles)))
    if not pred and not gold:
        return 1.0
    if not pred or not gold:
        return 0.0
    tp = len(pred & gold)
    precision, recall = tp / len(pred), tp / len(gold)
    return 0.0 if (precision + recall) == 0 else 2 * precision * recall / (precision + recall)


def cbr_outcome_alignment(pred_decision: str, analogous_cases: list) -> float:
    """Fraction of retrieved analogous cases whose gold decision agrees with
    the predicted decision — a cheap, non-LLM proxy reward component."""
    if not analogous_cases:
        return 0.5  # neutral prior when no exemplars are available
    agree = sum(1 for c in analogous_cases if c.get("gold_decision") == pred_decision)
    return agree / len(analogous_cases)


def composite_reward(strategy_json: dict, strategy_text: str, facts: str,
                     analogous_cases: list) -> dict:
    """Returns the scalar reward AND the diagnostic detail behind it.

    v1 computed per-criterion compliance and then discarded everything except
    score_pct, leaving the refinement step with nothing but aggregate numbers
    to act on. The `detail` payload below is what makes externally-grounded
    repair possible (see _build_refinement_feedback in §8)."""
    # compact (guideline-free) batched call — cheap enough to run up to 6x
    # per case (5 rollouts + 1 refinement) without ballooning judge calls
    rubric_res = rubric_batched_judge(strategy_text, facts, rubric_questions,
                                      include_guideline=False)
    rubric = rubric_res["score_pct"] / 100.0

    pred_arts = _flatten_articles(strategy_json.get("articulos_clave", []))
    gold_arts = [a for c in analogous_cases for a in c.get("gold_articles", [])]
    grounding = article_grounding_f1(pred_arts, gold_arts)

    pred_dec = strategy_json.get("decision_estimada", "")
    cbr_align = cbr_outcome_alignment(pred_dec, analogous_cases)

    total = (REWARD_WEIGHTS["rubric"] * rubric +
             REWARD_WEIGHTS["article_grounding"] * grounding +
             REWARD_WEIGHTS["cbr_alignment"] * cbr_align)

    # ---- error localisation (the part v1 threw away) ----
    failed_criteria = []
    for q in rubric_res.get("per_question", []):
        for c in q.get("criteria", []):
            if not c.get("cumple", False):
                failed_criteria.append({"question_id": q.get("question_id"),
                                        "name": c.get("name", ""),
                                        "weight_pct": c.get("weight_pct", 0.0)})
    # worst first: heaviest unmet criteria are the highest-leverage repairs
    failed_criteria.sort(key=lambda x: x["weight_pct"], reverse=True)

    _pred_set = {str(a).strip() for a in pred_arts}
    _gold_set = {str(a).strip() for a in gold_arts}
    cbr_decisions = [c.get("gold_decision", "") for c in analogous_cases]

    detail = {
        "failed_criteria": failed_criteria,
        "missing_articles": sorted(_gold_set - _pred_set),
        "unsupported_articles": sorted(_pred_set - _gold_set),
        "pred_decision": pred_dec,
        "cbr_decisions": cbr_decisions,
        "cbr_majority": max(set(cbr_decisions), key=cbr_decisions.count) if cbr_decisions else "",
    }
    return {"total": total, "rubric": rubric, "grounding": grounding,
            "cbr_align": cbr_align, "detail": detail}

print(f"  ✅  Reward model ready | weights={REWARD_WEIGHTS} | "
      f"judge={JUDGE_KEY} (fixed, out-of-family)\n")


In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# §8  ROLLOUT GENERATOR — Best-of-N stochastic sampling + self-refinement
# ──────────────────────────────────────────────────────────────────────────────
_STRATEGY_SYSTEM = (
    "Eres un abogado laboralista mexicano experto. Generas estrategias de "
    "litigio fundamentadas en la Ley Federal del Trabajo y en precedentes de "
    "la SCJN. Respondes ÚNICAMENTE en JSON válido con las claves: "
    "decision_estimada, probabilidad_exito, articulos_clave (lista), "
    "fortalezas (lista), debilidades (lista), estrategia (texto), "
    "advertencia_etica (texto). El valor de decision_estimada DEBE ser "
    'EXACTAMENTE uno de estos tres: "Procedente", "Improcedente", '
    '"Parcialmente Improcedente" (sin variaciones de mayúsculas ni texto '
    "adicional)."
)

def _build_context(facts: str, current_case_id: str = None) -> tuple:
    rag_chunks = retriever.invoke(facts)
    rag_context = "\n---\n".join(d.page_content for d in rag_chunks)
    # FIX: exclude the case under evaluation from its own CBR neighbor set
    # (prevents gold-label leakage — see retrieve_analogous_cases docstring).
    analogous = retrieve_analogous_cases(facts, current_case_id=current_case_id)
    cbr_context = "\n---\n".join(
        f"[Caso análogo {c['case_id']}] Decisión: {c.get('gold_decision','')} | "
        f"Artículos: {', '.join(c.get('gold_articles', []))}"
        for c in analogous)
    return rag_context, cbr_context, analogous


# CORRECTIVE FIX (post-hoc review after 1st full run on mistral-7b-v03-q4): the
# candidate occasionally abandons the required flat schema entirely (observed
# case: returned {"strategy": {"claims": [...], "evidence": [...]}} instead
# of the {decision_estimada, articulos_clave, ...} schema in _STRATEGY_SYSTEM).
# safe_json_extract() correctly parses this as valid, non-empty JSON, so it
# silently passed the old `if parsed:` truthy filter — but every downstream
# score that reads top-level keys (grounding, cbr_align) then reads defaults
# off a dict that structurally cannot contain them, producing a spuriously
# low reward that reflects a FORMAT violation, not content quality. A rollout
# in this state must never be treated as a valid, scorable candidate.
_REQUIRED_STRATEGY_KEYS = ("decision_estimada", "articulos_clave", "estrategia")

def _is_valid_strategy_schema(d) -> bool:
    if not isinstance(d, dict):
        return False
    if not all(k in d for k in _REQUIRED_STRATEGY_KEYS):
        return False
    return d.get("decision_estimada") in DECISION_LABELS


def generate_rollout(facts: str, rag_context: str, cbr_context: str,
                     temperature: float, top_p: float) -> tuple:
    prompt = (
        f"Hechos del caso:\n{facts}\n\n"
        f"Fragmentos normativos y de precedentes recuperados:\n{rag_context}\n\n"
        f"Casos análogos en el banco de referencia:\n{cbr_context}\n\n"
        "Genera la estrategia de litigio. Responde ÚNICAMENTE con un objeto "
        "JSON plano (sin anidar bajo ninguna clave contenedora) que tenga "
        "EXACTAMENTE estas claves de nivel superior: decision_estimada, "
        "probabilidad_exito, articulos_clave, fortalezas, debilidades, "
        "estrategia, advertencia_etica. No uses claves distintas ni "
        "estructuras alternativas (por ejemplo, NO uses \"claims\"/\"evidence\").\n\n"
        "Ejemplo de formato EXACTO esperado (usa SIEMPRE estas claves, "
        "adaptando el contenido al caso; este ejemplo es solo ilustrativo, "
        "no lo copies literalmente):\n"
        '{"decision_estimada": "Procedente", "probabilidad_exito": "Alta", '
        '"articulos_clave": ["LFT art. 47", "LFT art. 48"], '
        '"fortalezas": ["Ausencia de aviso de rescisión escrito"], '
        '"debilidades": ["Falta de testigos del despido"], '
        '"estrategia": "Se recomienda demandar indemnización constitucional '
        'con fundamento en...", '
        '"advertencia_etica": "Verificar prescripción de la acción (art. 517 LFT)."}'
    )
    try:
        text, usage = chat(llm, _STRATEGY_SYSTEM, prompt,
                           supports_system_role=MODEL_SPEC["supports_system_role"],
                           temperature=temperature, top_p=top_p, max_tokens=1000)
    except Exception as e:
        print(f"      ⚠️  generate_rollout failed [{type(e).__name__}]: {e or '(no message)'}")
        return None, {}, {}
    return text, safe_json_extract(text), usage


def generate_N(facts: str, n: int = N_ROLLOUTS, current_case_id: str = None) -> list:
    rag_context, cbr_context, analogous = _build_context(facts, current_case_id=current_case_id)
    # Phase 1: candidate generates all N rollouts (candidate is resident
    # throughout the run, so no load overhead here)
    raw_gens = []
    for _ in range(n):
        text, parsed, usage = generate_rollout(
            facts, rag_context, cbr_context, ROLLOUT_TEMP, ROLLOUT_TOP_P)
        if parsed and _is_valid_strategy_schema(parsed):
            raw_gens.append((text, parsed, usage))
        elif parsed:
            print(f"      ⚠️  rollout discarded: valid JSON but non-compliant "
                  f"schema (missing {[k for k in _REQUIRED_STRATEGY_KEYS if k not in parsed]} "
                  f"or invalid decision_estimada={parsed.get('decision_estimada')!r}).")
    # Phase 2: judge scores all N in one load/release cycle (LOW_RAM_MODE) —
    # avoids interleaving candidate/judge residency N times per case
    rollouts = []
    for text, parsed, usage in raw_gens:
        reward = composite_reward(parsed, text, facts, analogous)
        rollouts.append({"text": text, "json": parsed, "reward": reward, "usage": usage})
    if LOW_RAM_MODE:
        judge_mgr.release()
    return rollouts, analogous


def _build_refinement_feedback(reward: dict) -> str:
    """Converts reward diagnostics into LOCATED, actionable repair targets.

    Rationale (Tyen et al., 2024): LLMs can repair an error once its location
    is supplied, but cannot reliably locate it themselves. v1 asked the model
    to 'identify one concrete weakness' — precisely the operation that fails —
    while withholding the locations the reward model had already computed."""
    d = reward.get("detail", {})
    lines = []

    failed = d.get("failed_criteria", [])[:MAX_FEEDBACK_CRITERIA]
    if failed:
        lines.append("Criterios de la rúbrica NO cumplidos (los de mayor peso primero):")
        for c in failed:
            lines.append(f"  - [P{c['question_id']}, peso {c['weight_pct']}%] {c['name']}")

    missing = d.get("missing_articles", [])
    if missing:
        lines.append("Artículos citados en casos análogos que TU estrategia OMITE "
                     "(verifica si aplican y, si procede, fundaméntalos):")
        lines.append("  - " + "; ".join(missing[:12]))

    unsupported = d.get("unsupported_articles", [])
    if unsupported:
        lines.append("Artículos que citas pero que NO aparecen en los casos análogos "
                     "(justifica su pertinencia o sustitúyelos):")
        lines.append("  - " + "; ".join(unsupported[:12]))

    pred, majority = d.get("pred_decision", ""), d.get("cbr_majority", "")
    if majority and pred and pred != majority:
        lines.append(f"Tu decisión estimada ('{pred}') contradice la decisión "
                     f"mayoritaria de los casos análogos ('{majority}'). "
                     f"Justifica la divergencia con hechos distintivos del caso "
                     f"o corrige la estimación.")

    return "\n".join(lines) if lines else "(sin fallos localizados)"


def self_critique_and_revise(best_rollout: dict, facts: str, analogous: list) -> dict:
    """One reward-guided revision round.

    Two changes over v1, both driven by the pilot result (refined < best-of-N)
    and the self-correction literature:
      1. FEEDBACK: inject located failures (REFINEMENT_FEEDBACK_MODE
         ='grounded') instead of asking the model to locate its own errors.
      2. ACCEPTANCE GUARD: keep the revision only if its reward improves.
         Selection is by the REWARD MODEL, never by the gold label, so this
         is a deployable policy rather than oracle-guided filtering.

    The returned dict always carries `reward_refined_raw` and
    `refinement_applied` so §10 can report the UNGUARDED win-rate — the
    honest measurement of whether refinement helps — alongside the guarded
    reward actually used downstream."""
    if REFINEMENT_FEEDBACK_MODE == "grounded":
        feedback = _build_refinement_feedback(best_rollout["reward"])
        critique_prompt = (
            f"Hechos del caso:\n{facts}\n\n"
            f"Tu estrategia previa:\n{best_rollout['text']}\n\n"
            f"EVALUACIÓN EXTERNA — fallos localizados por el evaluador:\n"
            f"{feedback}\n\n"
            "Corrige ÚNICAMENTE los fallos señalados arriba, preservando "
            "intactas las partes de la estrategia que ya son correctas (en "
            "particular las fundamentaciones normativas acertadas). Si "
            "consideras que la estrategia previa ya es adecuada y una "
            "reescritura la empeoraría, devuélvela SIN CAMBIOS. "
            "Responde en el mismo formato JSON."
        )
    else:  # "intrinsic" — v1 behaviour, retained as the ablation arm
        critique_prompt = (
            f"Hechos del caso:\n{facts}\n\n"
            f"Tu estrategia previa:\n{best_rollout['text']}\n\n"
            f"Puntajes de recompensa recibidos: "
            f"{ {k: v for k, v in best_rollout['reward'].items() if k != 'detail'} }\n\n"
            "Identifica UNA debilidad concreta (grounding normativo insuficiente, "
            "incumplimiento de algún criterio de la rúbrica, o desalineación con "
            "los casos análogos) y reescribe la estrategia completa corrigiéndola. "
            "Responde en el mismo formato JSON."
        )

    base_total = best_rollout["reward"]["total"]
    try:
        text, usage = chat(llm, _STRATEGY_SYSTEM, critique_prompt,
                           supports_system_role=MODEL_SPEC["supports_system_role"],
                           temperature=REFINE_TEMP, max_tokens=1000)
    except Exception as e:
        print(f"      ⚠️  self_critique_and_revise failed [{type(e).__name__}]: "
              f"{e or '(no message)'} — keeping best-of-N unrefined.")
        out = dict(best_rollout)
        out.update({"reward_refined_raw": None, "refinement_applied": False})
        return out

    parsed = safe_json_extract(text)
    if not parsed or not _is_valid_strategy_schema(parsed):
        # FIX: a schema-non-compliant revision (valid JSON, wrong structure —
        # see _is_valid_strategy_schema note above generate_rollout) must be
        # discarded exactly like a failed parse, never promoted to
        # litigation_result — otherwise the "accepted" strategy silently
        # loses decision_estimada/articulos_clave and grounding/cbr_align
        # collapse to 0.0 for reasons unrelated to strategy quality.
        if parsed and not _is_valid_strategy_schema(parsed):
            print(f"      ⚠️  refinement discarded: valid JSON but non-compliant "
                  f"schema — keeping best-of-N unrefined.")
        out = dict(best_rollout)
        out.update({"reward_refined_raw": None, "refinement_applied": False})
        return out

    reward = composite_reward(parsed, text, facts, analogous)
    if LOW_RAM_MODE:
        judge_mgr.release()

    refined = {"text": text, "json": parsed, "reward": reward,
               "usage": usage, "reward_refined_raw": reward["total"]}

    if REFINEMENT_ACCEPT_GUARD and reward["total"] < base_total:
        # revision degraded the strategy — discard it, keep best-of-N
        out = dict(best_rollout)
        out.update({"reward_refined_raw": reward["total"], "refinement_applied": False})
        return out

    refined["refinement_applied"] = True
    return refined

print("✅  §8 rollout generator ready.\n")


In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# §9  RL-LITE LOOP — generate → score → select → self-refine → rescore
#     Supports: (a) PILOT_MODE timing/cost projection, (b) checkpoint/resume
# ──────────────────────────────────────────────────────────────────────────────
print("── §9  RL-lite Loop ────────────────────────────────────────────────────")
_full_case_bank = case_bank if case_bank else [{
    "case_id": "DEMO-001",
    "facts": "Trabajador demanda despido injustificado. El patrón no entregó aviso.",
    "gold_decision": "Procedente", "gold_articles": ["LFT art. 47", "LFT art. 48"],
}]

if PILOT_MODE:
    test_cases = _full_case_bank[:PILOT_N_CASES]
    print(f"  🧪  PILOT_MODE=True — running {len(test_cases)}/{len(_full_case_bank)} "
          f"cases to estimate per-case cost before the full run.\n"
          f"      Checkpoint: {CHECKPOINT_LOCAL}")
else:
    test_cases = _full_case_bank
    print(f"  ▶️  PILOT_MODE=False — running the full case bank "
          f"({len(test_cases)} cases).\n      Checkpoint: {CHECKPOINT_LOCAL}")

# ── resume: skip any case_id already present in the checkpoint for this mode ─
_ckpt_rows, _processed_ids = load_checkpoint(CHECKPOINT_LOCAL, CHECKPOINT_DRIVE)
_pending_cases = [c for c in test_cases
                 if c.get("case_id", "") not in _processed_ids]
if _processed_ids:
    print(f"  {len(_pending_cases)}/{len(test_cases)} case(s) remaining after resume.")

results_rows = list(_ckpt_rows)   # seed with whatever already completed
_since_last_flush = 0
# FIX: cases previously vanished silently (bare `continue`) whenever every
# rollout failed the schema-compliance check (§8) — the run kept going, but
# nothing recorded WHICH case_ids were dropped or why, which is unreportable
# for a paper (the factorial design's N-per-cell must be auditable). Track it.
_dropped_cases = []

for i, case in enumerate(_pending_cases):
    cid = case.get("case_id", f"CASE-{i+1:03d}")
    facts = case.get("facts", "")
    if not facts:
        _dropped_cases.append({"case_id": cid, "reason": "empty_facts", "detail": ""})
        continue
    print(f"  [{i+1:02d}/{len(_pending_cases)}] {cid} — generating {N_ROLLOUTS} rollouts...")
    t0 = _time.time()
    try:
        rollouts, analogous = generate_N(facts, N_ROLLOUTS, current_case_id=cid)  # FIX: no self-leakage
        if not rollouts:
            print(f"    ⚠️  {cid}: no valid rollouts parsed — skipping.")
            _dropped_cases.append({"case_id": cid, "reason": "no_valid_rollouts",
                                    "detail": "all 5 generations were either "
                                              "unparseable JSON or failed schema "
                                              "validation (_is_valid_strategy_schema)"})
            if LOW_RAM_MODE and judge_mgr.is_loaded:
                judge_mgr.release()  # generate_N may not have reached its own release
            continue
        best = max(rollouts, key=lambda r: r["reward"]["total"])
        refined = self_critique_and_revise(best, facts, analogous)
    except Exception as e:
        # A transient failure (e.g. OOM on one case) must not lose the run —
        # log the REAL error (type + traceback, not just str(e), which can be
        # empty for several exception classes) and continue; the checkpoint
        # already holds every prior case.
        print(f"    ❌  {cid} failed [{type(e).__name__}]: {e or '(no message)'}")
        print("    " + traceback.format_exc().replace("\n", "\n    "))
        if LOW_RAM_MODE and judge_mgr.is_loaded:
            judge_mgr.release()  # don't leak a loaded judge across cases on error
        continue
    _raw = refined.get("reward_refined_raw")
    _applied = refined.get("refinement_applied", False)
    # win = did the revision improve on best-of-N BEFORE the acceptance guard?
    # This is the scientific measurement; `reward_accepted` is what the
    # deployed policy actually returns.
    win = (_raw is not None) and (_raw > best["reward"]["total"])
    elapsed = _time.time() - t0
    row = {
        "arm": ARM, "model": MODEL_KEY, "arch": MODEL_ARCH, "quant": MODEL_QUANT,
        "case_id": cid, "n_rollouts": len(rollouts),
        "reward_best_of_n": best["reward"]["total"],
        "reward_refined_raw": _raw,
        "reward_accepted": refined["reward"]["total"],
        "refinement_win": win,
        "refinement_applied": _applied,
        "feedback_mode": REFINEMENT_FEEDBACK_MODE,
        "accept_guard": REFINEMENT_ACCEPT_GUARD,
        "gold_decision": case.get("gold_decision", ""),
        "pred_decision": refined["json"].get("decision_estimada", ""),
        "correct_decision": case.get("gold_decision", "") == refined["json"].get("decision_estimada", ""),
        "elapsed_sec": elapsed,
        "final_strategy_json": refined["json"],
    }
    results_rows.append(row)
    _since_last_flush += 1
    _raw_s = f"{_raw:.3f}" if _raw is not None else "n/a"
    print(f"    reward best-of-{N_ROLLOUTS}={best['reward']['total']:.3f} → "
          f"refined_raw={_raw_s} → accepted={refined['reward']['total']:.3f} | "
          f"win={win} applied={_applied} | {elapsed:.1f}s")

    if _since_last_flush >= CHECKPOINT_EVERY:
        append_checkpoint(row, CHECKPOINT_LOCAL, CHECKPOINT_DRIVE)
        _since_last_flush = 0
    if _since_last_flush > 0 and (i + 1) == len(_pending_cases):
        append_checkpoint(row, CHECKPOINT_LOCAL, CHECKPOINT_DRIVE)  # final flush

results_df = pd.DataFrame(results_rows)
_n_dropped = len(_dropped_cases)
_compliance_rate = 100.0 * (len(test_cases) - _n_dropped) / max(len(test_cases), 1)
print(f"\n✅  §9 complete: {len(results_df)}/{len(test_cases)} case(s) in this pass "
      f"({_n_dropped} dropped, schema-compliance rate={_compliance_rate:.1f}%).\n")
if _dropped_cases:
    print(f"  ⚠️  Dropped case_ids: {[d['case_id'] for d in _dropped_cases]}")
    _drop_path = f"/content/dropped_cases_{ARM_SUFFIX}.json"
    with open(_drop_path, "w", encoding="utf-8") as f:
        json.dump({"arm": ARM, "model": MODEL_KEY, "n_total": len(test_cases),
                   "n_dropped": _n_dropped, "compliance_rate_pct": _compliance_rate,
                   "dropped": _dropped_cases}, f, indent=2, ensure_ascii=False)
    try:
        _shutil_model.copy2(_drop_path, os.path.join(MODELS_DIR_DRIVE, "..",
                             os.path.basename(_drop_path)))
    except Exception as _e:
        print(f"  (could not mirror drop log to Drive: {_e})")
    print(f"  Drop log saved: {_drop_path} — report this compliance rate in "
          f"the paper's Methods/Limitations (per-model, per-arm).")

# ── pilot → full-run cost projection ──────────────────────────────────────────
if PILOT_MODE and not results_df.empty:
    _mean_s   = results_df["elapsed_sec"].mean()
    _n_full   = len(_full_case_bank)
    _n_arms   = len(MODEL_REGISTRY)  # 5 candidates share this cost structure
    _proj_one = _mean_s * _n_full
    _proj_all = _proj_one * _n_arms
    print("  ── Pilot → full-run projection ──────────────────────────────────")
    print(f"  Mean time/case (this arm)      : {_mean_s:.1f} s "
          f"(n={len(results_df)} pilot cases)")
    print(f"  Projected time — this arm      : {_proj_one/60:.1f} min "
          f"for {_n_full} cases")
    print(f"  Projected time — all {_n_arms} arms      : {_proj_all/60:.1f} min "
          f"({_proj_all/3600:.2f} h) if every arm behaves similarly")
    print(f"  ⚠️  Colab free-tier sessions typically cap around 12h (and can\n"
          f"      disconnect earlier); if the per-arm projection above exceeds\n"
          f"      ~2-3 h, checkpoint/resume (already active) will let you\n"
          f"      continue across sessions — just re-run §9 with PILOT_MODE\n"
          f"      unchanged and it will pick up from {CHECKPOINT_DRIVE}.")
    print(f"\n  ➡️  If this cost is acceptable:\n"
          f"      1. set PILOT_MODE = False in §2\n"
          f"      2. RE-RUN §2  (required — PILOT_MODE, CHECKPOINT_LOCAL and\n"
          f"         CHECKPOINT_DRIVE are all defined there; skipping §2 leaves\n"
          f"         the checkpoint pointing at the pilot file and §9 would\n"
          f"         resume from these {len(results_df)} pilot cases)\n"
          f"      3. re-run §9 (no need to reload §0-§8/§6b — models and\n"
          f"         indexes stay in memory)\n"
          f"      The pilot checkpoint ({CHECKPOINT_LOCAL}) is a separate file\n"
          f"      from the full-run checkpoint, so nothing needs deleting.\n"
          f"      TIP: run §10 on this pilot first — it is fast and verifies\n"
          f"      §10-§13 work before you commit hours to the full run.\n")


In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# §10  EVALUATION — Tier-1/2/3 metrics + bootstrap CIs + RL-specific metrics
# ──────────────────────────────────────────────────────────────────────────────
print("── §10  Evaluation ────────────────────────────────────────────────────")

def bootstrap_ci(values, n_resamples=2000, seed=42):
    if len(values) == 0:
        return {"lo": float("nan"), "hi": float("nan")}
    rng = np.random.default_rng(seed)
    vals = np.array(values)
    means = [rng.choice(vals, size=len(vals), replace=True).mean() for _ in range(n_resamples)]
    return {"lo": round(float(np.percentile(means, 2.5)), 4),
           "hi": round(float(np.percentile(means, 97.5)), 4)}

if not results_df.empty:
    y_true = [DECISION_LABEL2ID.get(d, -1) for d in results_df["gold_decision"]]
    y_pred = [DECISION_LABEL2ID.get(d, -1) for d in results_df["pred_decision"]]
    valid_mask = [(t != -1 and p != -1) for t, p in zip(y_true, y_pred)]
    yt = [t for t, m in zip(y_true, valid_mask) if m]
    yp = [p for p, m in zip(y_pred, valid_mask) if m]
    if yt:
        acc = accuracy_score(yt, yp)
        macro_f1 = f1_score(yt, yp, average="macro", zero_division=0)
        mcc = matthews_corrcoef(yt, yp) if len(set(yt)) > 1 else 0.0
        print(f"  Decision accuracy : {acc:.4f}  95% CI {bootstrap_ci([int(a==b) for a,b in zip(yt,yp)])}")
        print(f"  Macro F1          : {macro_f1:.4f}")
        print(f"  MCC               : {mcc:.4f}")

    reward_best = results_df["reward_best_of_n"].tolist()
    reward_acc  = results_df["reward_accepted"].tolist()
    print(f"\n  ── Reward ──")
    print(f"  Best-of-{N_ROLLOUTS} (selection only) : mean={np.mean(reward_best):.4f} "
          f"95% CI {bootstrap_ci(reward_best)}")
    print(f"  Accepted (deployed policy)  : mean={np.mean(reward_acc):.4f} "
          f"95% CI {bootstrap_ci(reward_acc)}")

    # ── Refinement effect: report the UNGUARDED comparison, which is the
    # actual scientific question (does self-refinement help?). The guarded
    # 'accepted' reward above is the engineering policy and would mask a
    # negative effect by construction.
    _paired = results_df.dropna(subset=["reward_refined_raw"])
    print(f"\n  ── Self-refinement effect (mode='{REFINEMENT_FEEDBACK_MODE}', "
          f"guard={REFINEMENT_ACCEPT_GUARD}, judge={JUDGE_KEY}) ──")
    if _paired.empty:
        print("  No parseable revisions produced — refinement effect undefined.")
    else:
        _deltas = (_paired["reward_refined_raw"] - _paired["reward_best_of_n"]).tolist()
        _win_rate = float((_paired["reward_refined_raw"] > _paired["reward_best_of_n"]).mean())
        _applied_rate = float(results_df["refinement_applied"].mean())
        print(f"  n paired                : {len(_deltas)}")
        print(f"  Unguarded win-rate      : {_win_rate:.2%} "
              f"(revision > best-of-{N_ROLLOUTS}, BEFORE the acceptance guard)")
        print(f"  Mean Δreward            : {np.mean(_deltas):+.4f} "
              f"95% CI {bootstrap_ci(_deltas)}")
        print(f"  Revisions kept by guard : {_applied_rate:.2%}")
        try:
            from scipy.stats import wilcoxon
            if len(_deltas) >= 6 and any(d != 0 for d in _deltas):
                _stat, _p = wilcoxon(_deltas)
                print(f"  Wilcoxon signed-rank    : W={_stat:.1f}, p={_p:.4f}"
                      f"  ({'significant' if _p < 0.05 else 'n.s.'} at α=0.05)")
            else:
                print("  Wilcoxon signed-rank    : skipped (n<6 or all Δ=0)")
        except Exception as _e:
            print(f"  Wilcoxon signed-rank    : unavailable ({_e})")
        _m = np.mean(_deltas)
        if _m < 0:
            print("  ⚠️  INTERPRETATION: refinement DEGRADES the strategy on "
                  "average. Consistent with Huang et al. (2024) on intrinsic "
                  "self-correction. If mode='grounded' still yields Δ<0, the "
                  "defensible thesis claim is that the RL-lite gain comes "
                  "from best-of-N selection, with the revision step reported "
                  "as a negative result — not silently hidden by the guard.")
        else:
            print("  ✅  INTERPRETATION: refinement improves the strategy on "
                  "average; the guard then removes the residual harmful cases.")

    _csv = f"/content/rl_benchmark_results_{RUN_SUFFIX}.csv"
    results_df.drop(columns=["final_strategy_json"]).to_csv(_csv, index=False, encoding="utf-8")
    print(f"\n  Saved: {_csv}")
print("✅  §10 complete.\n")


In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# §11  LITIGATION STRATEGY — final structured RAG + CBR + RL output (demo case)
# ──────────────────────────────────────────────────────────────────────────────
print("── §11  Litigation Strategy (demo case) ───────────────────────────────")
_demo_facts = (
    "Un empleado con 5 años de antigüedad fue despedido verbalmente por su "
    "supervisor sin causa justificada. El patrón no entregó aviso escrito de "
    "rescisión, no invocó artículos de la LFT y no pagó partes proporcionales. "
    "El trabajador demanda indemnización constitucional, salarios caídos y "
    "prima de antigüedad conforme a la Ley Federal del Trabajo.")

_demo_rollouts, _demo_analogous = generate_N(_demo_facts, N_ROLLOUTS)
_demo_best = max(_demo_rollouts, key=lambda r: r["reward"]["total"])
litigation_result = self_critique_and_revise(_demo_best, _demo_facts, _demo_analogous)

print(f"  Reward final           : {litigation_result['reward']}")
print(f"  Probabilidad de éxito  : {litigation_result['json'].get('probabilidad_exito','')}")
print(f"  Artículos clave        : {', '.join(litigation_result['json'].get('articulos_clave', []))}")
print("  Fortalezas:")
[print(f"    • {x}") for x in litigation_result["json"].get("fortalezas", [])]
print("  Debilidades:")
[print(f"    • {x}") for x in litigation_result["json"].get("debilidades", [])]
print(f"  Estrategia: {litigation_result['json'].get('estrategia','')[:200]}")
print(f"  ⚠️  {litigation_result['json'].get('advertencia_etica','')}")

_lit = f"/content/litigation_strategy_{RUN_SUFFIX}.json"
with open(_lit, "w", encoding="utf-8") as f:
    json.dump({"arm": ARM, "model": MODEL_KEY, "judge": JUDGE_KEY,
              "demo_facts": _demo_facts, "reward": litigation_result["reward"],
              "litigation_strategy": litigation_result["json"]},
             f, indent=2, ensure_ascii=False)
print(f"\n  ✅  Saved: {_lit}")


# ──────────────────────────────────────────────────────────────────────────────
# §11b  RUBRIC EVALUATION — 20-question, JUDGE_MODEL only
# ──────────────────────────────────────────────────────────────────────────────
print("\n── §11b  Rubric Evaluation ─────────────────────────────────────────────")
rubric_summary = {}
if rubric_questions:
    # FULL version (guidelines included) — the formal, thesis-reportable
    # score; called once on the final refined strategy, not per rollout
    rubric_result = rubric_batched_judge(litigation_result["text"], _demo_facts,
                                         rubric_questions, include_guideline=True)
    if LOW_RAM_MODE:
        judge_mgr.release()
    rubric_summary = {"final_score_pct": rubric_result["score_pct"], "judge": JUDGE_KEY}
    print(f"  Final rubric score: {rubric_result['score_pct']:.1f}%  (judge={JUDGE_KEY}, fixed)")
    _rj = f"/content/rubric_evaluation_results_{RUN_SUFFIX}.json"
    with open(_rj, "w", encoding="utf-8") as f:
        json.dump({"arm": ARM, "model": MODEL_KEY, "judge": JUDGE_KEY,
                  "result": rubric_result}, f, indent=2, ensure_ascii=False)
    print(f"  ✅  Saved: {_rj}")
else:
    print("  Rubric skipped (no questions loaded).")


In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# §12  VISUALISATION + §12b QUALITATIVE SAMPLE
# ──────────────────────────────────────────────────────────────────────────────
print("\n── §12  Visualisation & Export ────────────────────────────────────────")
if not results_df.empty:
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    _raw_mean = results_df["reward_refined_raw"].dropna().mean()
    axes[0].bar(["Best-of-N", "Refined (raw)", "Accepted"],
               [results_df["reward_best_of_n"].mean(),
                _raw_mean if not np.isnan(_raw_mean) else 0.0,
                results_df["reward_accepted"].mean()],
               color=["#9DC3E6", "#C00000", "#1F3864"])
    axes[0].set_ylim(0, 1.0)
    axes[0].set_title(f"Mean Composite Reward ({ARM})")
    win_counts = results_df["refinement_win"].value_counts()
    axes[1].bar(["Refinement hurt", "Refinement helped"],
               [win_counts.get(False, 0), win_counts.get(True, 0)],
               color=["#C00000", "#375623"])
    axes[1].set_title(f"Unguarded refinement outcome "
                      f"(mode={REFINEMENT_FEEDBACK_MODE})")
    plt.suptitle(f"RL + LLM Legal v1 — [{ARM}]", fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.savefig(f"/content/rl_benchmark_plots_{RUN_SUFFIX}.png", dpi=150)
    plt.show()

    # §12b — 5-case qualitative sample for thesis narrative
    _qual = results_df.nlargest(min(5, len(results_df)), "reward_accepted")[
        ["case_id", "gold_decision", "pred_decision", "reward_accepted",
         "refinement_applied", "final_strategy_json"]
    ].to_dict(orient="records")
    with open(f"/content/qualitative_sample_{RUN_SUFFIX}.json", "w", encoding="utf-8") as f:
        json.dump(_qual, f, indent=2, ensure_ascii=False)
    print(f"  ✅  Qualitative sample ({len(_qual)} cases) saved.")
print("✅  §12 complete.\n")


In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# §13  SAVE ALL ARTIFACTS TO DRIVE — ARM-namespaced subdirectory
# ──────────────────────────────────────────────────────────────────────────────
print("── §13  Saving Artifacts to Drive ─────────────────────────────────────")
_OUT = os.path.join(DRIVE_ROOT, "results", ARM)
os.makedirs(_OUT, exist_ok=True)
_saved, _failed = [], []
for fn in [f"rl_benchmark_results_{RUN_SUFFIX}.csv",
          f"rl_benchmark_plots_{RUN_SUFFIX}.png",
          f"litigation_strategy_{RUN_SUFFIX}.json",
          f"rubric_evaluation_results_{RUN_SUFFIX}.json",
          f"qualitative_sample_{RUN_SUFFIX}.json"]:
    src = f"/content/{fn}"
    if os.path.exists(src):
        try:
            _shutil_model.copy2(src, os.path.join(_OUT, fn)); _saved.append(fn)
        except Exception as e:
            _failed.append((fn, str(e)))
for _src, _dst in [(_faiss_local, FAISS_INDEX_PATH), (_cbr_local, CBR_INDEX_PATH)]:
    if not os.path.exists(_src):
        continue
    _dst_stale = os.path.exists(_dst) and not embedding_cache_matches(_dst)
    if not os.path.exists(_dst) or _dst_stale:
        try:
            if _dst_stale:
                _shutil_model.rmtree(_dst)
                print(f"  ♻️  Replacing stale Drive index at {_dst} "
                      f"(built with a different embedding model) with the "
                      f"current one — prevents the FAISS dimension-mismatch "
                      f"crash from recurring in future sessions/arms.")
            _shutil_model.copytree(_src, _dst)
            _saved.append(os.path.basename(_dst))
        except Exception as e:
            _failed.append((os.path.basename(_dst), str(e)))
print(f"  Saved to {_OUT}: {_saved}")
if _failed:
    print(f"  Failed: {_failed}")
print("✅  §13 complete.\n")


In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# §14  CASE ANALYSIS MODULE — Reusable PhD Experiment API
# ──────────────────────────────────────────────────────────────────────────────
_CASE_REGISTRY: dict = {}

def run_case_analysis(case_id: str, facts: str, save: bool = True, verbose: bool = True) -> dict:
    """Full RL-lite litigation analysis (generate → reward → refine) + rubric
    evaluation on any case scenario, callable from a new Colab cell without
    modifying prior sections."""
    if not facts or not facts.strip():
        raise ValueError("facts cannot be empty.")
    if verbose:
        print(f"\n{'═'*70}\n  CASE ANALYSIS: {case_id}  [{MODEL_KEY}]\n{'═'*70}")
    rollouts, analogous = generate_N(facts, N_ROLLOUTS, current_case_id=case_id)  # FIX: no self-leakage
    if not rollouts:
        raise RuntimeError(f"No valid rollouts for case '{case_id}'.")
    best = max(rollouts, key=lambda r: r["reward"]["total"])
    refined = self_critique_and_revise(best, facts, analogous)
    rubric = (rubric_batched_judge(refined["text"], facts, rubric_questions,
                                   include_guideline=True)
             if rubric_questions else {})
    if LOW_RAM_MODE and rubric_questions:
        judge_mgr.release()
    result = {"case_id": case_id, "facts": facts, "model": MODEL_KEY,
              "judge": JUDGE_KEY, "strategy": refined["json"],
              "reward": refined["reward"], "rubric": rubric,
              "timestamp": _time.strftime("%Y-%m-%dT%H:%M:%S")}
    _CASE_REGISTRY[case_id] = result
    if save:
        _cases_dir = os.path.join(DRIVE_ROOT, "case_analyses", MODEL_KEY)
        os.makedirs(_cases_dir, exist_ok=True)
        with open(os.path.join(_cases_dir, f"{case_id}.json"), "w", encoding="utf-8") as f:
            json.dump(result, f, indent=2, ensure_ascii=False)
    if verbose:
        print(f"  Reward: {result['reward']}")
        print(f"  Estrategia: {result['strategy'].get('estrategia','')[:200]}")
    return result


def compare_cases() -> pd.DataFrame:
    rows = [{"case_id": k, "reward_total": v["reward"]["total"],
            "rubric_pct": v["rubric"].get("score_pct", float("nan"))}
           for k, v in _CASE_REGISTRY.items()]
    return pd.DataFrame(rows)

print("✅  §14 Case Analysis Module ready. Usage:")
print("     result = run_case_analysis('CASE-MAT-01', 'Trabajadora embarazada...')")
print("     compare_cases()")
print("\n================ RL + LLM Legal v1 — pipeline loaded ================\n")
